# Module 4: Genetic Algorithm - Route Optimization

**WanderWise+ TPOS-Aligned Implementation**

Optimizes daily tourism routes using Genetic Algorithm following the TPOS paper:
*"Travel Planning Optimization System Employing Genetic Algorithms with Multiple Parameters"*  
Rusu & Alexandrescu, ICSTCC 2024

## Key TPOS Features Implemented:

1. **Population**: 100 chromosomes per day
2. **Selection**: Tournament selection (k=2)
3. **Crossover**: Single-point (70% probability)
4. **Mutation**: Swap mutation (20% probability)
5. **Elitism**: Best solution preserved
6. **Fitness**: `1 / (1 + Δ)` where Δ = sum of penalties
7. **Stopping**: Max 50 generations or plateau detection

See `ga_parameter_tuning_TPOS.md` for full specification.

## Fitness Function: Reward + Penalty Model

### The Problem with Pure Penalty Models

The original TPOS formula `fitness = 1 / (1 + Δ)` only minimizes penalties:
- **1-POI route**: Δ = 2 → fitness = 0.33 ✓ ("good" score)
- **6-POI route**: Δ = 170 → fitness = 0.006 ✗ ("bad" score)

**Result:** GA learns that shorter routes = better fitness! Routes collapse to 1 POI.

### The Fix: Add POI Value Reward

```
fitness = POI_VALUE_SUM / (1 + Δ)
```

Where:
- **POI_VALUE_SUM** = sum of normalized_popularity (REWARD)
- **Δ** = sum of penalties (PENALTY)

**Now:**
- **1-POI route**: poi_value = 0.9, Δ = 2 → fitness = 0.9/3 = **0.30**
- **6-POI route**: poi_value = 5.4, Δ = 170 → fitness = 5.4/171 = **0.032**

Still needs better penalty scaling, but now longer routes can compete!

### Penalty Components (Scaled for Goa):

All penalty multipliers scaled down from TPOS to match Goa data:

1. **Distance Penalty** (per consecutive POI pair):
   - `penalty = walking_distance × 1` (TPOS: ×10,000)
   - Typical Goa inter-POI: 5-15km

2. **User Preference Penalty** (per POI):
   - `penalty = (100 - preference%) × 1` (TPOS: ×10)
   - Note: Already rewarded via poi_value_sum
   - Can be set to 0 if desired

3. **Hard Violation Penalty**:
   - `1,000 per gene` (TPOS: 10,000)
   - Closed POIs or visit exceeds open hours

4. **Must-See Location Penalty**:
   - `(β - α) × 100` (TPOS: ×1,000)

5. **Restaurant Placement Penalty**:
   - `100` (TPOS: 1,000)

### Additional Safeguards:

✅ **Minimum route length enforced**: Crossover cannot produce routes < MIN_POIS_PER_ROUTE  
✅ **POI value reward**: Longer routes with high-WPI POIs favored  
✅ **Penalty scaling**: Adjusted for Goa distance/popularity ranges  

### Comparison:

| Model | Formula | 1-POI Route | 6-POI Route | Winner |
|-------|---------|-------------|-------------|--------|
| TPOS Pure Penalty | 1/(1+Δ) | 0.33 | 0.006 | 1-POI ❌ |
| WanderWise Reward+Penalty | value/(1+Δ) | 0.30 | 0.032+ | 6-POI ✅ |


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import random
import requests
from math import radians, sin, cos, asin, sqrt
from dataclasses import dataclass
from typing import List, Tuple

print('✅ Imports loaded')

✅ Imports loaded


In [21]:
# GA Parameters (Based on TPOS paper - ga_parameter_tuning_TPOS.md)
# Reference: Travel Planning Optimization System Employing Genetic Algorithms
# Rusu & Alexandrescu, ICSTCC 2024

POPULATION_SIZE = 100           # Fixed initial population per day
MAX_GENERATIONS = 50            # Stops on fitness plateau or max iterations
CROSSOVER_RATE = 0.7            # Single-point crossover probability (TPOS spec)
MUTATION_RATE = 0.2             # Swap mutation probability (TPOS spec)
TOURNAMENT_SIZE = 2             # Two random individuals compete (TPOS spec)
ELITE_COUNT = 3                # Best solution preserved (TPOS spec)
EARLY_STOPPING_THRESHOLD = 20   # Stop if no improvement for N generations

# Chromosome Configuration (TPOS approach)
INITIAL_CHROMOSOME_LENGTH = 6   # Initial genes (trimmed dynamically based on time)
MIN_POIS_PER_ROUTE = 4          # Minimum POIs in a route
MAX_POIS_PER_ROUTE = 15         # Maximum POIs in a route

# TPOS Penalty Values (ADJUSTED FOR GOA DATA SCALE)
# Note: Original TPOS multipliers (10000, 10) were calibrated for Paris data
# We scale these down for Goa's distance/popularity ranges while keeping TPOS formula
HARD_VIOLATION_PENALTY = 1000   # Per gene: closed POI or exceeds open hours (10000→1000)
USER_PREFERENCE_PENALTY_MULTIPLIER = 1.0   # (100 - preference%) × 1 (10→1)
DISTANCE_PENALTY_MULTIPLIER = 1.0          # walking_distance × 1 (10000→1)
MUST_SEE_PENALTY_MULTIPLIER = 100          # (β - α) × 100 (1000→100)
RESTAURANT_PENALTY = 100                   # Incorrect restaurant count/placement (1000→100)

# Time Configuration (TPOS spec)
TOUR_START_TIME = "09:00"       # Day start time (TPOS spec)
TOUR_END_TIME = "18:00"         # Day end/trim time (TPOS spec ~04:00 PM)
LUNCH_START_TIME = "12:00"
LUNCH_END_TIME = "13:30"
DAILY_TIME_BUDGET_HOURS = 9    # 09:00 to 16:00 = 7 hours

# POI Default Values
DEFAULT_VISIT_DURATION_MIN = 60
DEFAULT_OPENING_TIME = "09:00"
DEFAULT_CLOSING_TIME = "18:00"

# Visit Duration by Location Type (TPOS Section 7)
VISIT_DURATIONS = {
    'museum': 90,
    'park': 60,
    'shopping_mall': 120,
    'zoo': 120,
    'waterfall':120,
    'default': 60
}

# Travel Speed (Goa road conditions)
AVERAGE_SPEED_KM_H = 30

print(f"⚙️ GA Configuration (TPOS Paper Aligned - Goa Data Scale):")
print(f"   Population: {POPULATION_SIZE} chromosomes")
print(f"   Generations: {MAX_GENERATIONS} (with plateau detection)")
print(f"   Crossover: {CROSSOVER_RATE*100}% (single-point)")
print(f"   Mutation: {MUTATION_RATE*100}% (swap only)")
print(f"   Selection: Tournament (k={TOURNAMENT_SIZE})")
print(f"   Elitism: {ELITE_COUNT} best preserved")
print(f"\n🧬 Chromosome Configuration:")
print(f"   Initial length: {INITIAL_CHROMOSOME_LENGTH} genes")
print(f"   Range: {MIN_POIS_PER_ROUTE}-{MAX_POIS_PER_ROUTE} POIs")
print(f"\n⏰ Time Configuration:")
print(f"   Tour: {TOUR_START_TIME} - {TOUR_END_TIME} ({DAILY_TIME_BUDGET_HOURS}h)")
print(f"   Lunch: {LUNCH_START_TIME} - {LUNCH_END_TIME}")
print(f"\n⚠️ TPOS Penalty System (SCALED FOR GOA):")
print(f"   Hard violation: {HARD_VIOLATION_PENALTY} per gene")
print(f"   Distance: {DISTANCE_PENALTY_MULTIPLIER}x km (adjusted from 10000x)")
print(f"   User pref: {USER_PREFERENCE_PENALTY_MULTIPLIER}x diff (adjusted from 10x)")
print(f"   Must-see: {MUST_SEE_PENALTY_MULTIPLIER}x gap (adjusted from 1000x)")
print(f"\n📊 Penalty Scaling Rationale:")
print(f"   TPOS multipliers calibrated for Paris data (different scales)")
print(f"   Adjusted for Goa: distances (0-50km), popularity (0-1)")
print(f"   Formula structure unchanged: fitness = 1/(1+Δ)")


⚙️ GA Configuration (TPOS Paper Aligned - Goa Data Scale):
   Population: 100 chromosomes
   Generations: 50 (with plateau detection)
   Crossover: 70.0% (single-point)
   Mutation: 20.0% (swap only)
   Selection: Tournament (k=2)
   Elitism: 3 best preserved

🧬 Chromosome Configuration:
   Initial length: 6 genes
   Range: 4-15 POIs

⏰ Time Configuration:
   Tour: 09:00 - 18:00 (9h)
   Lunch: 12:00 - 13:30

⚠️ TPOS Penalty System (SCALED FOR GOA):
   Hard violation: 1000 per gene
   Distance: 1.0x km (adjusted from 10000x)
   User pref: 1.0x diff (adjusted from 10x)
   Must-see: 100x gap (adjusted from 1000x)

📊 Penalty Scaling Rationale:
   TPOS multipliers calibrated for Paris data (different scales)
   Adjusted for Goa: distances (0-50km), popularity (0-1)
   Formula structure unchanged: fitness = 1/(1+Δ)


In [22]:
# ============================================================
# DEFAULT CONSTRAINTS (fallbacks until real-time data available)
# ============================================================

DEFAULT_CONSTRAINTS = {
    "opening_time": "09:00",
    "closing_time": "18:00",
    "visit_duration_min": 60,
    "physical_intensity": "medium",   # low / medium / high
    "crowd_factor": 0.5,              # 0.0 (empty) -> 1.0 (crowded)
    "best_time_start": "09:00",
    "best_time_end": "18:00",
}

# Per-type fallback overrides
TYPE_DEFAULTS = {
    "beach": {
        "visit_duration_min": 90,
        "best_time_start": "15:30",
        "best_time_end": "18:30",
        "physical_intensity": "medium",
    },
    "waterfall": {
        "visit_duration_min": 120,
        "best_time_start": "10:00",
        "best_time_end": "1:00",
        "physical_intensity": "high",
    },
    "historical": {
        "visit_duration_min": 75,
        "best_time_start": "09:00",
        "best_time_end": "17:00",
        "physical_intensity": "medium",
    },
    "religious": {
        "visit_duration_min": 45,
        "best_time_start": "09:00",
        "best_time_end": "16:00",
        "physical_intensity": "low",
    },
    "museum": {
        "visit_duration_min": 90,
        "best_time_start": "10:00",
        "best_time_end": "16:00",
        "physical_intensity": "low",
    },
}

## 2. Load Data

In [23]:
df = pd.read_csv('module2_clustered_normalized_places.csv')
print(f'Loaded {len(df)} places')
print(f'Days: {df["day"].nunique()}')
print(f'WPI range: {df["normalized_popularity"].min():.3f} - {df["normalized_popularity"].max():.3f}')
df.head()

Loaded 464 places
Days: 4
WPI range: 0.447 - 1.000


,name,address,rating,user_ratings_total,place_id,types,taluka,latitude,longitude,Unnamed: 9,Unnamed: 10,categories,cluster,day,normalized_google_rating,weighted_rating,quantity_factor,popularity_score,cluster_max_popularity,normalized_popularity
0,Fort Aguada,"Fort Aguada Rd, Aguada Fort Area, Candolim, Go...",4.2,106573.0,ChIJa72MxnXBvzsRHHtpszB2g6M,"tourist_attraction, point_of_interest, establi...",Bardez (Mapusa),15.492252,73.773746,NaN,NaN,"['adventure', 'historical']",0,1,0.800,0.800000,1.000000,0.880000,0.908466,0.968666
1,Dudhsagar Falls,"Sonauli, Goa 403410, India",4.6,30982.0,ChIJQ4srFBimvzsRJQI7KOdIlP0,"tourist_attraction, point_of_interest, establi...",Sanguem,15.314438,74.314307,NaN,NaN,"['adventure', 'nature']",2,3,0.900,0.899998,1.000000,0.939999,0.939999,1.000000
2,Velsao Beach,"9V3M+QG9, Consua, Goa 403712, India",4.3,4971.0,ChIJY6h2EJ-3vzsRB_haiJ7F51w,"tourist_attraction, point_of_interest, establi...",Mormugao (Vasco),15.354423,73.883864,NaN,NaN,['beaches'],3,4,0.825,0.825003,0.910455,0.859184,0.919589,0.934313
3,Cabo de Rama Fort,"3WQC+GJ8, Taluka Cabo da Rama, Canacona, Goa 4...",4.4,15290.0,ChIJy1w3MFFJvjsRW9cTbAchLdw,"tourist_attraction, point_of_interest, establi...",Canacona,15.088785,73.921593,NaN,NaN,"['adventure', 'historical']",1,2,0.850,0.849999,0.937868,0.885147,0.939999,0.941647
4,Harvalem Waterfalls,"Rudreshwar Colony, Kudne, Goa 403505, India",4.3,9127.0,ChIJVUbAVEy9vzsRS6uKBv4zNmI,"tourist_attraction, point_of_interest, establi...",Sattari (Valpoi),15.550773,74.026470,NaN,NaN,['nature'],2,3,0.825,0.825002,0.881823,0.847730,0.939999,0.901842


In [24]:
# Check how many POIs qualify as must-see
must_see_count = sum(1 for _, row in df.iterrows() 
                     if row['normalized_popularity'] >= 0.9)
total = len(df)
print(f"Must-see POIs: {must_see_count}/{total} ({must_see_count/total*100:.1f}%)")
# Aim for roughly 10-20% of your pool
# If too few → lower threshold to 0.80
# If too many → raise to 0.90

Must-see POIs: 46/464 (9.9%)


In [25]:
def infer_primary_type(name: str, raw_type: str = "") -> str:
    txt = f"{name} {raw_type}".lower()
    if any(k in txt for k in ["waterfall", "falls", "dudhsagar", "arvalem"]):
        return "waterfall"
    if any(k in txt for k in ["beach", "coast", "shore", "palolem", "baga", "anjuna", "candolim"]):
        return "beach"
    if any(k in txt for k in ["fort", "heritage", "ruins", "historical", "tower"]):
        return "historical"
    if any(k in txt for k in ["temple", "church", "cathedral", "basilica", "chapel", "mosque"]):
        return "religious"
    if "museum" in txt:
        return "museum"
    return "default"

def apply_default_constraints(name: str, raw_type: str = "", existing: dict | None = None) -> dict:
    existing = existing or {}
    ptype = infer_primary_type(name, raw_type)
    merged = dict(DEFAULT_CONSTRAINTS)
    merged.update(TYPE_DEFAULTS.get(ptype, {}))
    # existing real value wins
    merged.update({k: v for k, v in existing.items() if v not in [None, "", np.nan]})
    merged["primary_type"] = ptype
    return merged

## 3. POI Class

In [26]:
class POI:
    def __init__(self, name, lat, lon, normalized_popularity, 
                 place_id=None, rating=3.0, user_ratings_total=0):
        self.name = name
        self.lat = lat
        self.lon = lon
        self.normalized_popularity = normalized_popularity
        self.place_id = place_id or name
        self.rating = rating
        self.user_ratings_total = user_ratings_total
        self.opening_time = DEFAULT_OPENING_TIME
        self.closing_time = DEFAULT_CLOSING_TIME
        self.visit_duration_min = DEFAULT_VISIT_DURATION_MIN
    
    def __repr__(self):
        return f"POI({self.name}, WPI={self.normalized_popularity:.3f})"

def load_pois_for_day(df, day):
    day_data = df[df['day'] == day]
    pois = []
    for _, row in day_data.iterrows():
        poi = POI(
            name=row['name'],
            lat=row['latitude'],
            lon=row['longitude'],
            normalized_popularity=row['normalized_popularity'],
            place_id=row.get('place_id', row['name']),
            rating=row['rating'],
            user_ratings_total=row['user_ratings_total']
        )
        pois.append(poi)
        defaults = apply_default_constraints(
    name=row["name"],
    raw_type=str(row.get("types", "")),
    existing={
        "opening_time": row.get("opening_time"),
        "closing_time": row.get("closing_time"),
        "visit_duration_min": row.get("visit_duration_min"),
        "physical_intensity": row.get("physical_intensity"),
        "crowd_factor": row.get("crowd_factor"),
        "best_time_start": row.get("best_time_start"),
        "best_time_end": row.get("best_time_end"),
    }
)

    poi.opening_time = defaults["opening_time"]
    poi.closing_time = defaults["closing_time"]
    poi.visit_duration_min = int(defaults["visit_duration_min"])
    poi.type = defaults["primary_type"]
    poi.physical_intensity = defaults["physical_intensity"]
    poi.crowd_factor = float(defaults["crowd_factor"])
    poi.best_time_start = defaults["best_time_start"]
    poi.best_time_end = defaults["best_time_end"]
    return pois

print('✅ POI class defined')

✅ POI class defined


## 4. Distance & Time Functions

In [27]:
# ================================
# Distance, time, and OSRM matrix helpers
# ================================

OSRM_BASE_URL = "http://localhost:5000"
USE_OSRM = True

# Current-day matrices (set once per day before GA runs)
CURRENT_DISTANCE_KM = None
CURRENT_DURATION_MIN = None
CURRENT_POI_INDEX = {}

def haversine_distance(coord1, coord2):
    """
    Great-circle distance between two coordinates.

    Args:
        coord1: (lat, lon)
        coord2: (lat, lon)

    Returns:
        Distance in km
    """
    lat1, lon1 = map(radians, coord1)
    lat2, lon2 = map(radians, coord2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return 6371.0 * c

def time_to_minutes(time_str):
    """
    Convert HH:MM -> total minutes.
    """
    h, m = map(int, time_str.split(":"))
    return h * 60 + m

def minutes_to_time(total_minutes):
    """
    Convert total minutes -> HH:MM (24h wrap).
    """
    total_minutes = int(total_minutes)
    h = (total_minutes // 60) % 24
    m = total_minutes % 60
    return f"{h:02d}:{m:02d}"

def build_osrm_matrices(pois, osrm_url=OSRM_BASE_URL):
    """
    Build distance and duration matrices from OSRM Table API.

    Returns:
        (distance_km_matrix, duration_min_matrix)
    """
    if len(pois) == 0:
        return np.zeros((0, 0)), np.zeros((0, 0))

    # OSRM requires lon,lat
    coords = ";".join([f"{p.lon},{p.lat}" for p in pois])
    url = f"{osrm_url}/table/v1/driving/{coords}?annotations=distance,duration"

    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    if data.get("code") != "Ok":
        raise RuntimeError(f"OSRM Table API failed: {data}")

    distance_km = np.array(data["distances"], dtype=float) / 1000.0
    duration_min = np.array(data["durations"], dtype=float) / 60.0
    return distance_km, duration_min

def build_haversine_matrices(pois):
    """
    Fallback matrix builder using haversine distance and constant speed.
    """
    n = len(pois)
    distance_km = np.zeros((n, n), dtype=float)
    duration_min = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            d = haversine_distance((pois[i].lat, pois[i].lon), (pois[j].lat, pois[j].lon))
            distance_km[i, j] = d
            duration_min[i, j] = (d / AVERAGE_SPEED_KM_H) * 60.0

    return distance_km, duration_min

def prepare_day_matrices(pois, use_osrm=USE_OSRM):
    """
    Prepare global matrices once per day, before GA starts.
    """
    global CURRENT_DISTANCE_KM, CURRENT_DURATION_MIN, CURRENT_POI_INDEX

    # Use object identity index so duplicate place_id values do not break mapping
    CURRENT_POI_INDEX = {id(p): i for i, p in enumerate(pois)}

    if use_osrm:
        try:
            CURRENT_DISTANCE_KM, CURRENT_DURATION_MIN = build_osrm_matrices(pois)
            print(f"✅ OSRM matrices ready: {len(pois)}x{len(pois)}")
            return
        except Exception as e:
            print(f"⚠️ OSRM unavailable, fallback to haversine. Reason: {e}")

    CURRENT_DISTANCE_KM, CURRENT_DURATION_MIN = build_haversine_matrices(pois)
    print(f"📍 Haversine matrices ready: {len(pois)}x{len(pois)}")

def get_cached_distance(poi1, poi2):
    """
    Distance lookup (km) from prepared matrix.
    """
    i = CURRENT_POI_INDEX[id(poi1)]
    j = CURRENT_POI_INDEX[id(poi2)]
    return float(CURRENT_DISTANCE_KM[i, j])

def calculate_travel_time(poi1, poi2):
    """
    Duration lookup (minutes) from prepared matrix.
    """
    i = CURRENT_POI_INDEX[id(poi1)]
    j = CURRENT_POI_INDEX[id(poi2)]
    return float(CURRENT_DURATION_MIN[i, j])

print('✅ OSRM integration ready')

✅ OSRM integration ready


In [ ]:
# def select_top_pois_for_optimization(pois, max_candidates=50):
#     """
#     Pre-filter POIs to top N by WPI score before passing to GA.
    
#     This serves two purposes:
#     1. Stay within OSRM Table API limits (usually 100 max)
#     2. Speed up GA by reducing search space
    
#     Args:
#         pois: All POIs from cluster
#         max_candidates: Maximum POIs to pass to GA (default 50)
    
#     Returns:
#         Top POIs by WPI score (deduplicated)
#     """
#     # Remove duplicates by place_id or name+coords
#     seen = set()
#     unique_pois = []
    
#     for poi in pois:
#         # Use place_id if available, else name+coords
#         key = poi.place_id if hasattr(poi, 'place_id') else f"{poi.name}_{poi.lat}_{poi.lon}"
#         if key not in seen:
#             seen.add(key)
#             unique_pois.append(poi)
    
#     dedup_count = len(pois) - len(unique_pois)
#     if dedup_count > 0:
#         print(f"   🔧 Removed {dedup_count} duplicate POIs")
    
#     # Sort by WPI score (normalized_popularity)
#     sorted_pois = sorted(unique_pois, key=lambda p: p.normalized_popularity, reverse=True)
    
#     # Select top N
#     if len(sorted_pois) <= max_candidates:
#         print(f"   📍 Passing all {len(sorted_pois)} unique POIs to GA")
#         return sorted_pois
#     else:
#         selected = sorted_pois[:max_candidates]
#         wpi_min = min(p.normalized_popularity for p in selected)
#         wpi_max = max(p.normalized_popularity for p in selected)
        
#         print(f"   📍 Selected top {len(selected)}/{len(sorted_pois)} POIs by WPI")
#         print(f"      WPI range: {wpi_min:.3f} - {wpi_max:.3f}")
#         print(f"      (Filtered out {len(sorted_pois) - len(selected)} lower-value POIs)")
        
#         return selected

# print('✅ POI pre-filtering function defined')

✅ POI pre-filtering function defined


In [28]:
from difflib import SequenceMatcher

# ── Types that should never appear as tourist stops ───────────────────────────
EXCLUDED_TYPES = {
    'lodging', 'hotel', 'resort', 'motel', 'hostel', 'guest_house',
    'restaurant', 'cafe', 'bar', 'food', 'bakery', 'meal_takeaway',
    'meal_delivery', 'night_club', 'liquor_store',
    'store', 'shop', 'supermarket', 'grocery_or_supermarket', 'shopping_mall',
    'hospital', 'pharmacy', 'doctor', 'dentist', 'health',
    'gas_station', 'parking', 'car_rental', 'taxi_stand',
    'bank', 'atm', 'finance', 'insurance_agency',
    'real_estate_agency', 'moving_company', 'storage',
}

# ── Fuzzy name similarity ─────────────────────────────────────────────────────
def _names_are_similar(name1: str, name2: str, threshold: float = 0.82) -> bool:
    """
    True if two place names are suspiciously similar.
    Catches cases like 'Palolem Beach' vs 'Palolem Beach, GOA'
    where Google assigns different place_ids.
    """
    n1 = name1.lower().strip()
    n2 = name2.lower().strip()

    # Exact match after normalisation
    if n1 == n2:
        return True

    # One is a prefix/suffix of the other  e.g. "Palolem Beach" in "Palolem Beach, GOA"
    if n1 in n2 or n2 in n1:
        return True

    # Sequence similarity
    return SequenceMatcher(None, n1, n2).ratio() > threshold


# ── Proximity check ───────────────────────────────────────────────────────────
def _coords_are_close(lat1, lon1, lat2, lon2, threshold_km: float = 0.1) -> bool:
    """
    True if two POIs are within threshold_km of each other.
    Uses a fast flat-earth approximation — fine for sub-km distances.
    """
    dlat = abs(lat1 - lat2) * 111.0          # 1° lat ≈ 111 km
    dlon = abs(lon1 - lon2) * 111.0 * 0.85   # rough cos(lat) factor for Goa
    return (dlat ** 2 + dlon ** 2) ** 0.5 < threshold_km


# ── Type filter ───────────────────────────────────────────────────────────────
def _poi_is_excluded(poi) -> bool:
    """Return True if the POI is a non-tourist venue (hotel, restaurant, etc.)"""
    poi_type = getattr(poi, 'type', '') or ''
    poi_type = poi_type.lower().strip()

    # Direct match
    if poi_type in EXCLUDED_TYPES:
        return True

    # Name-based heuristic for cases where type is missing/wrong
    name_lower = poi.name.lower()
    name_keywords = [
        'hotel', 'resort', 'hostel', 'inn ', ' inn', 'lodge', 'villa ',
        'restaurant', ' cafe', 'bistro', 'shack', 'bar ', ' bar',
        'hospital', 'clinic', 'pharmacy', 'atm', 'bank',
    ]
    if any(kw in name_lower for kw in name_keywords):
        return True

    return False


# ── Category diversity ────────────────────────────────────────────────────────
def _enforce_category_diversity(pois, max_per_type: int = 4) -> list:
    """
    Prevent one category from dominating the candidate pool.
    e.g. if there are 20 beaches, cap them at max_per_type
    so the GA also sees forts, temples, waterfalls, etc.

    Within each category, keep the highest-WPI ones.
    """
    type_counts = {}
    diverse = []
    deferred = []   # overflow — added back at the end if slots remain

    for poi in pois:   # already sorted by WPI descending coming in
        poi_type = getattr(poi, 'type', 'unknown') or 'unknown'
        count = type_counts.get(poi_type, 0)

        if count < max_per_type:
            diverse.append(poi)
            type_counts[poi_type] = count + 1
        else:
            deferred.append(poi)

    # Fill remaining slots from overflow (still in WPI order)
    diverse.extend(deferred)
    return diverse


# ── Main function ─────────────────────────────────────────────────────────────
def select_top_pois_for_optimization(pois, max_candidates: int = 50) -> list:
    """
    Pre-filter POIs to the best N candidates before passing to the GA.

    Steps applied in order:
    1. Remove non-tourist venues (hotels, restaurants, etc.)
    2. Deduplicate by place_id
    3. Fuzzy deduplicate by name similarity + proximity
       (catches 'Palolem Beach' vs 'Palolem Beach, GOA')
    4. Sort by WPI score descending
    5. Enforce category diversity (cap per type)
    6. Select top N

    Args:
        pois:           All POIs loaded for this day/cluster
        max_candidates: Hard cap passed to GA (default 50)

    Returns:
        Filtered, deduplicated, diverse list of POIs — best candidates first
    """
    original_count = len(pois)

    # ── Step 1: Remove excluded types ─────────────────────────────────────────
    tourist_pois = [p for p in pois if not _poi_is_excluded(p)]
    excluded_count = original_count - len(tourist_pois)

    if excluded_count > 0:
        print(f"   🚫 Removed {excluded_count} non-tourist venues "
              f"(hotels, restaurants, etc.)")

    if not tourist_pois:
        print("   ⚠️  No tourist POIs remaining after type filter!")
        return []

    # ── Step 2: Deduplicate by place_id ───────────────────────────────────────
    seen_ids = set()
    id_deduped = []

    for poi in tourist_pois:
        key = getattr(poi, 'place_id', None) or f"{poi.name}_{poi.lat:.5f}_{poi.lon:.5f}"
        if key not in seen_ids:
            seen_ids.add(key)
            id_deduped.append(poi)

    id_dup_count = len(tourist_pois) - len(id_deduped)
    if id_dup_count > 0:
        print(f"   🔧 Removed {id_dup_count} exact duplicate place_ids")

    # ── Step 3: Fuzzy name + proximity deduplication ──────────────────────────
    # Sort by WPI first so we keep the higher-rated version of near-duplicates
    id_deduped.sort(key=lambda p: p.normalized_popularity, reverse=True)

    fuzzy_deduped = []
    for candidate in id_deduped:
        is_near_duplicate = False

        for accepted in fuzzy_deduped:
            name_match = _names_are_similar(candidate.name, accepted.name)
            coord_match = _coords_are_close(
                candidate.lat, candidate.lon,
                accepted.lat, accepted.lon
            )

            # Flag as duplicate if EITHER names are similar OR coords are very close
            # (Catches both naming variants and same-spot multiple listings)
            if name_match or coord_match:
                is_near_duplicate = True
                break

        if not is_near_duplicate:
            fuzzy_deduped.append(candidate)

    fuzzy_dup_count = len(id_deduped) - len(fuzzy_deduped)
    if fuzzy_dup_count > 0:
        print(f"   🔧 Removed {fuzzy_dup_count} near-duplicate POIs "
              f"(similar names / same location)")

    # ── Step 4: Sort by WPI descending (already done above, kept for clarity) ─
    fuzzy_deduped.sort(key=lambda p: p.normalized_popularity, reverse=True)

    # ── Step 5: Category diversity cap ────────────────────────────────────────
    # Cap per type BEFORE the top-N cut so no single type monopolises the pool
    diverse_pois = _enforce_category_diversity(fuzzy_deduped, max_per_type=4)

    # ── Step 6: Select top N ──────────────────────────────────────────────────
    if len(diverse_pois) <= max_candidates:
        final = diverse_pois
    else:
        final = diverse_pois[:max_candidates]

    # ── Summary ───────────────────────────────────────────────────────────────
    type_breakdown = {}
    for p in final:
        t = getattr(p, 'type', 'unknown') or 'unknown'
        type_breakdown[t] = type_breakdown.get(t, 0) + 1

    wpi_min = min(p.normalized_popularity for p in final)
    wpi_max = max(p.normalized_popularity for p in final)

    print(f"\n   ✅ Final candidate pool: {len(final)} POIs "
          f"(from {original_count} original)")
    print(f"      WPI range : {wpi_min:.3f} – {wpi_max:.3f}")
    print(f"      Types     : {dict(sorted(type_breakdown.items(), key=lambda x: -x[1]))}")
    if len(final) < 6:
        print(f"   ⚠️  WARNING: Only {len(final)} candidates — "
              f"day may have too few stops. Consider loosening filters "
              f"or merging with adjacent cluster.")

    return final


print('✅ Improved POI pre-filtering function defined')

✅ Improved POI pre-filtering function defined


## 5. Fitness Function

In [ ]:
# class RouteEvaluation:
#     """Stores route evaluation results with reward + penalty model."""
#     def __init__(self):
#         # Reward component (positive)
#         self.poi_value_sum = 0.0         # Sum of normalized_popularity
        
#         # TPOS penalty components (negative) - adjusted for Goa data scale
#         self.distance_penalty = 0.0      # walking_distance × 1
#         self.user_pref_penalty = 0.0     # (100 - preference%) × 1
#         self.hard_violation_penalty = 0.0  # 1000 per gene
#         self.must_see_penalty = 0.0      # (β - α) × 100
#         self.restaurant_penalty = 0.0    # Restaurant count/placement
        
#         # Metrics for tracking
#         self.total_travel_time_min = 0.0
#         self.total_visit_time_min = 0.0
#         self.total_distance_km = 0.0
#         self.closed_poi_count = 0
#         self.lunch_invasion_count = 0
#         self.overtime_minutes = 0
#         self.total_time_min = 0.0
#         self.timeline = []
        
#         # Combined fitness
#         self.delta = 0.0      # Total penalty sum
#         self.fitness = 0.0    # Final fitness score
        
#         # Legacy fields for backward compatibility with debug cells
#         self.travel_penalty = 0.0
#         self.constraint_penalty = 0.0

# def evaluate_fitness(route, start_time=TOUR_START_TIME):
#     """
#     Evaluate route fitness with REWARD + PENALTY model.
    
#     FITNESS = POI_VALUE_SUM / (1 + Δ)
    
#     Where:
#     - POI_VALUE_SUM = sum of normalized_popularity (REWARD for including POIs)
#     - Δ = sum of penalties (PENALTY for distance, violations, etc.)
    
#     This ensures:
#     - Longer routes with high-value POIs are rewarded
#     - Penalties still discourage bad route characteristics
#     - 1-POI routes cannot trivially dominate
    
#     Penalties (scaled for Goa):
#     - Distance: walking_distance × 1
#     - User preference: (100 - preference%) × 1 per POI
#     - Hard violation: 1000 per closed/invalid POI
#     - Restaurant: incorrect count/placement
    
#     Higher fitness = better solution
#     """
#     eval_result = RouteEvaluation()
#     if not route:
#         eval_result.delta = float('inf')
#         eval_result.fitness = 0.0
#         return eval_result
    
#     current_time_min = time_to_minutes(start_time)
#     tour_start_min = time_to_minutes(TOUR_START_TIME)
#     lunch_start_min = time_to_minutes(LUNCH_START_TIME)
#     lunch_end_min = time_to_minutes(LUNCH_END_TIME)
#     daily_budget_min = DAILY_TIME_BUDGET_HOURS * 60
    
#     # Calculate POI value reward (positive contribution)
#     eval_result.poi_value_sum = sum(poi.normalized_popularity for poi in route)
    
#     # Simulate tour and accumulate penalties
#     for i, poi in enumerate(route):
#         # Distance penalty - applied per consecutive pair
#         if i > 0:
#             travel_dist_km = get_cached_distance(route[i-1], poi)
#             travel_time_min = calculate_travel_time(route[i-1], poi)
            
#             # Distance penalty (scaled for Goa: ×1)
#             eval_result.distance_penalty += travel_dist_km * DISTANCE_PENALTY_MULTIPLIER
            
#             current_time_min += travel_time_min
#             eval_result.total_travel_time_min += travel_time_min
#             eval_result.total_distance_km += travel_dist_km
        
#         # User preference penalty
#         # Note: We already reward high popularity via poi_value_sum
#         # This penalty is for TPOS compatibility, but can be set to 0 if desired
#         user_preference_pct = poi.normalized_popularity * 100
#         pref_penalty = (100 - user_preference_pct) * USER_PREFERENCE_PENALTY_MULTIPLIER
#         eval_result.user_pref_penalty += pref_penalty
        
#         # Check lunch invasion
#         if lunch_start_min <= current_time_min < lunch_end_min:
#             eval_result.lunch_invasion_count += 1
#             current_time_min = max(current_time_min, lunch_end_min)
        
#         arrival_time = current_time_min
        
#         # Hard violation penalty - location closed or visit exceeds hours
#         poi_opening_min = time_to_minutes(poi.opening_time)
#         poi_closing_min = time_to_minutes(poi.closing_time)
        
#         if current_time_min < poi_opening_min or current_time_min >= poi_closing_min:
#             eval_result.closed_poi_count += 1
#             eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
#             if current_time_min < poi_opening_min:
#                 current_time_min = poi_opening_min
        
#         # Visit POI
#         visit_start = current_time_min
#         visit_end = current_time_min + poi.visit_duration_min
#         current_time_min = visit_end
#         eval_result.total_visit_time_min += poi.visit_duration_min
        
#         # Check if visit exceeds closing time
#         if visit_end > poi_closing_min:
#             eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
        
#         eval_result.timeline.append((poi, minutes_to_time(int(arrival_time)),
#                                      minutes_to_time(int(visit_start)), 
#                                      minutes_to_time(int(visit_end))))
    
#     # Calculate total time
#     eval_result.total_time_min = current_time_min - tour_start_min
#     eval_result.overtime_minutes = max(0, eval_result.total_time_min - daily_budget_min)
    
#     # Restaurant penalty (placeholder)
    
#     # Calculate total penalty (Δ)
#     eval_result.delta = (
#         eval_result.distance_penalty +
#         eval_result.user_pref_penalty +
#         eval_result.hard_violation_penalty +
#         eval_result.must_see_penalty +
#         eval_result.restaurant_penalty
#     )
    
#     # Legacy fields for debug cells
#     eval_result.travel_penalty = eval_result.distance_penalty
#     eval_result.constraint_penalty = (
#         eval_result.hard_violation_penalty +
#         eval_result.must_see_penalty +
#         eval_result.restaurant_penalty
#     )
    
#     # FITNESS = REWARD / (1 + PENALTY)
#     # This balances POI value (reward) against penalties
#     # - High-value POIs increase numerator
#     # - Penalties increase denominator
#     # - Longer routes with good POIs can beat short routes
#     eval_result.fitness = eval_result.poi_value_sum / (1.0 + eval_result.delta)
    
#     return eval_result

# print('✅ Fitness function with POI value reward defined')


✅ Fitness function with POI value reward defined


In [ ]:
# class RouteEvaluation:
#     """Stores route evaluation results."""
#     def __init__(self):
#         # Reward components
#         self.poi_value_sum = 0.0
#         self.time_utilization_bonus = 0.0   # Reward for filling the day
#         self.route_length_bonus = 0.0       # Reward for visiting more places

#         # Penalty components
#         self.distance_penalty = 0.0
#         self.hard_violation_penalty = 0.0
#         self.undertime_penalty = 0.0        # Penalise finishing too early
#         self.overtime_penalty = 0.0         # Penalise going over time budget

#         # Metrics
#         self.total_travel_time_min = 0.0
#         self.total_visit_time_min = 0.0
#         self.total_distance_km = 0.0
#         self.closed_poi_count = 0
#         self.lunch_invasion_count = 0
#         self.overtime_minutes = 0
#         self.undertime_minutes = 0
#         self.total_time_min = 0.0
#         self.timeline = []

#         # Final scores
#         self.delta = 0.0
#         self.fitness = 0.0

#         # Legacy compatibility
#         self.travel_penalty = 0.0
#         self.constraint_penalty = 0.0
#         self.user_pref_penalty = 0.0
#         self.must_see_penalty = 0.0
#         self.restaurant_penalty = 0.0


# # ── Tunable constants ──────────────────────────────────────────────────────────
# # Penalties
# DISTANCE_PENALTY_WEIGHT     = 0.5    # per km — keep low so GA doesn't just pick nearby duds
# HARD_VIOLATION_PENALTY      = 50.0   # per closed/overtime POI — high enough to deter
# LUNCH_INVASION_PENALTY      = 10.0   # per invasion — moderate
# UNDERTIME_PENALTY_PER_MIN   = 0.15   # per minute under budget — gentle push to fill day
# OVERTIME_PENALTY_PER_MIN    = 0.20   # per minute over budget — steeper to stop overrun

# # Rewards
# TIME_UTILIZATION_WEIGHT     = 2.0    # scales the time-fill bonus
# ROUTE_LENGTH_BONUS_PER_POI  = 0.05   # small bonus per POI beyond minimum

# # Tour window
# TOUR_START_MINUTES = time_to_minutes(TOUR_START_TIME)    # e.g. 540 (09:00)
# TOUR_END_MINUTES   = time_to_minutes(TOUR_END_TIME)      # e.g. 1080 (18:00)
# DAILY_BUDGET_MIN   = TOUR_END_MINUTES - TOUR_START_MINUTES  # e.g. 480


# def evaluate_fitness(route, start_time=TOUR_START_TIME):
#     """
#     Fitness function designed to:

#     1. REWARD visiting high-quality POIs
#     2. REWARD filling the available time window (9am–6pm)
#     3. REWARD longer routes (up to the time limit)
#     4. PENALISE distance between consecutive stops
#     5. PENALISE visiting closed POIs or exceeding closing time
#     6. PENALISE lunch window invasions
#     7. PENALISE finishing significantly under budget (lazy routes)
#     8. PENALISE going over the daily time budget

#     Formula:
#         fitness = (poi_value_sum + time_utilization_bonus + route_length_bonus)
#                   / (1 + delta)

#     Where delta = sum of all penalties.
#     """
#     eval_result = RouteEvaluation()

#     if not route:
#         eval_result.fitness = 0.0
#         return eval_result

#     current_time_min = time_to_minutes(start_time)
#     lunch_start_min  = time_to_minutes(LUNCH_START_TIME)
#     lunch_end_min    = time_to_minutes(LUNCH_END_TIME)

#     # ── 1. Simulate the tour ──────────────────────────────────────────────────
#     for i, poi in enumerate(route):

#         # Travel from previous POI
#         if i > 0:
#             dist_km       = get_cached_distance(route[i - 1], poi)
#             travel_min    = calculate_travel_time(route[i - 1], poi)

#             eval_result.total_distance_km    += dist_km
#             eval_result.total_travel_time_min += travel_min
#             eval_result.distance_penalty     += dist_km * DISTANCE_PENALTY_WEIGHT

#             current_time_min += travel_min

#         # Auto-skip lunch window: if arrival lands in lunch, wait it out
#         if lunch_start_min <= current_time_min < lunch_end_min:
#             eval_result.lunch_invasion_count += 1
#             eval_result.hard_violation_penalty += LUNCH_INVASION_PENALTY
#             current_time_min = lunch_end_min   # resume after lunch

#         arrival_time_min = current_time_min

#         # Opening hours check
#         poi_open_min  = time_to_minutes(poi.opening_time)
#         poi_close_min = time_to_minutes(poi.closing_time)

#         if current_time_min < poi_open_min:
#             # Arrive before opening — wait (no penalty, realistic behaviour)
#             current_time_min = poi_open_min

#         elif current_time_min >= poi_close_min:
#             # Arrive after closing — hard violation
#             eval_result.closed_poi_count += 1
#             eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
#             # Still simulate visiting so timeline is complete
#             # GA will learn to avoid this

#         visit_start_min = current_time_min
#         visit_end_min   = current_time_min + poi.visit_duration_min

#         # Check if visit runs past closing time
#         if visit_end_min > poi_close_min and current_time_min < poi_close_min:
#             eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY * 0.5  # softer

#         eval_result.total_visit_time_min += poi.visit_duration_min
#         current_time_min = visit_end_min

#         eval_result.timeline.append((
#             poi,
#             minutes_to_time(int(arrival_time_min)),
#             minutes_to_time(int(visit_start_min)),
#             minutes_to_time(int(visit_end_min))
#         ))

#     # ── 2. Time accounting ────────────────────────────────────────────────────
#     eval_result.total_time_min = current_time_min - TOUR_START_MINUTES

#     # Overtime: finished after TOUR_END_TIME
#     if current_time_min > TOUR_END_MINUTES:
#         eval_result.overtime_minutes = current_time_min - TOUR_END_MINUTES
#         eval_result.overtime_penalty = (
#             eval_result.overtime_minutes * OVERTIME_PENALTY_PER_MIN
#         )

#     # Undertime: finished more than 60 min before TOUR_END_TIME
#     # 60-min buffer is fine (packing up, dinner plans etc.)
#     UNDERTIME_BUFFER_MIN = 60
#     time_remaining = TOUR_END_MINUTES - current_time_min
#     if time_remaining > UNDERTIME_BUFFER_MIN:
#         eval_result.undertime_minutes = time_remaining - UNDERTIME_BUFFER_MIN
#         eval_result.undertime_penalty = (
#             eval_result.undertime_minutes * UNDERTIME_PENALTY_PER_MIN
#         )

#     # ── 3. Reward components ──────────────────────────────────────────────────

#     # Base POI value
#     eval_result.poi_value_sum = sum(poi.normalized_popularity for poi in route)

#     # Time utilisation bonus: reward routes that fill the day well
#     # Ratio of time used vs available, capped at 1.0 (don't reward overtime)
#     time_used   = min(eval_result.total_time_min, DAILY_BUDGET_MIN)
#     time_ratio  = time_used / DAILY_BUDGET_MIN           # 0.0 → 1.0
#     eval_result.time_utilization_bonus = time_ratio * TIME_UTILIZATION_WEIGHT

#     # Route length bonus: small reward per POI beyond MIN_POIS_PER_ROUTE
#     extra_pois = max(0, len(route) - MIN_POIS_PER_ROUTE)
#     eval_result.route_length_bonus = extra_pois * ROUTE_LENGTH_BONUS_PER_POI

#     # ── 4. Assemble penalty delta ─────────────────────────────────────────────
#     eval_result.delta = (
#         eval_result.distance_penalty
#         + eval_result.hard_violation_penalty
#         + eval_result.undertime_penalty
#         + eval_result.overtime_penalty
#     )

#     # Legacy fields
#     eval_result.travel_penalty      = eval_result.distance_penalty
#     eval_result.constraint_penalty  = eval_result.hard_violation_penalty
#     eval_result.user_pref_penalty   = 0.0   # removed: double-counted with poi_value_sum
#     eval_result.must_see_penalty    = 0.0   # placeholder — add if you implement must-see
#     eval_result.restaurant_penalty  = 0.0   # removed: not applicable to your dataset

#     # ── 5. Final fitness ──────────────────────────────────────────────────────
#     total_reward = (
#         eval_result.poi_value_sum
#         + eval_result.time_utilization_bonus
#         + eval_result.route_length_bonus
#     )

#     eval_result.fitness = total_reward / (1.0 + eval_result.delta)

#     return eval_result


# print('✅ Improved fitness function defined')
# print(f'   Tour window : {TOUR_START_TIME} – {TOUR_END_TIME} ({DAILY_BUDGET_MIN} min)')
# print(f'   Undertime penalty : {UNDERTIME_PENALTY_PER_MIN}/min (buffer: 60 min)')
# print(f'   Overtime penalty  : {OVERTIME_PENALTY_PER_MIN}/min')
# print(f'   Hard violation    : {HARD_VIOLATION_PENALTY} per event')
# print(f'   Distance weight   : {DISTANCE_PENALTY_WEIGHT} per km')

✅ Improved fitness function defined
   Tour window : 09:00 – 18:00 (540 min)
   Undertime penalty : 0.15/min (buffer: 60 min)
   Overtime penalty  : 0.2/min
   Hard violation    : 50.0 per event
   Distance weight   : 0.5 per km


In [29]:
# ============================================================
# ENHANCED FITNESS FUNCTION — WanderWise+ Goa
# ============================================================
# New features vs previous version:
#   1. Best-time scheduling  — beaches in evening, waterfalls morning
#   2. Waterfall rules       — max 1 per day, penalise 2+
#   3. Must-see reward       — bonus for including high-WPI anchors
#   4. Proximity cluster     — reward visiting neighbours of must-see
#   5. Fatigue model         — too many physical stops costs extra
# ============================================================

# ── Time window constants ─────────────────────────────────────────────────────
TOUR_START_MINUTES = time_to_minutes(TOUR_START_TIME)     # 540  (09:00)
TOUR_END_MINUTES   = time_to_minutes(TOUR_END_TIME)       # 1080 (18:00)
DAILY_BUDGET_MIN   = TOUR_END_MINUTES - TOUR_START_MINUTES  # 480

MORNING_END_MIN    = time_to_minutes("12:00")   # 720
AFTERNOON_END_MIN  = time_to_minutes("15:00")   # 900
EVENING_START_MIN  = time_to_minutes("15:30")   # 930

# ── Penalty weights ───────────────────────────────────────────────────────────
DISTANCE_PENALTY_WEIGHT       = 0.5    # per km
HARD_VIOLATION_PENALTY        = 50.0   # closed POI / exceeded hours
LUNCH_INVASION_PENALTY        = 10.0   # visiting during 12:00-13:30
UNDERTIME_PENALTY_PER_MIN     = 0.15   # finishing >60 min early
OVERTIME_PENALTY_PER_MIN      = 0.20   # going past TOUR_END
WRONG_TIME_PENALTY            = 8.0    # beach at 9am, waterfall at 5pm
EXTRA_WATERFALL_PENALTY       = 30.0   # each waterfall beyond the 1st
PHYSICAL_FATIGUE_PENALTY      = 5.0    # each strenuous stop beyond 2/day

# ── Reward weights ────────────────────────────────────────────────────────────
TIME_UTILIZATION_WEIGHT       = 2.0    # 0→2.0 as day fills up
ROUTE_LENGTH_BONUS_PER_POI    = 0.05   # per POI beyond MIN_POIS_PER_ROUTE
MUST_SEE_REWARD               = 0.3    # per must-see POI in route
MUST_SEE_NEIGHBOUR_REWARD     = 0.15   # per neighbour of must-see in route
CORRECT_TIME_REWARD           = 0.1    # beach at right time, waterfall at right time

# ── Must-see WPI threshold ────────────────────────────────────────────────────
# POIs above this WPI are treated as "must-see anchors" for reward purposes.
# Tune this: 0.85 = top ~15% of your dataset; 0.90 = top ~10%
MUST_SEE_WPI_THRESHOLD        = 0.85

# ── Type classification helpers ───────────────────────────────────────────────
# Keywords matched against poi.name.lower() and poi.type.lower()
# Add/remove keywords to match your actual dataset labels.

WATERFALL_KEYWORDS   = {'waterfall', 'falls', 'dhudh', 'dudhsagar', 'vajra',
                        'vazra', 'sakla', 'arvalem'}

BEACH_KEYWORDS       = {'beach', 'praia', 'coast', 'shore', 'sea', 'palolem',
                        'vagator', 'baga', 'anjuna', 'calangute', 'candolim',
                        'morjim', 'arambol', 'colva', 'benaulim', 'butterfly',
                        'agonda', 'patnem', 'backwater'}

STRENUOUS_KEYWORDS   = {'waterfall', 'falls', 'trek', 'fort', 'hill', 'ghat',
                        'chorla', 'peak', 'climb', 'hike'}

# HERITAGE_KEYWORDS    = {'basilica', 'cathedral', 'church', 'chapel', 'temple',
#                         'mosque', 'masjid', 'fort', 'museum', 'archaeological',
#                         'heritage', 'palace', 'ruins', 'tower', 'monastery'}


def _classify_poi(poi) -> set:
    """
    Return a set of category tags for a POI.
    Uses both poi.type and poi.name for robustness.
    """
    tags = set()
    name  = poi.name.lower()
    ptype = (getattr(poi, 'type', '') or '').lower()
    combined = name + ' ' + ptype

    if any(kw in combined for kw in WATERFALL_KEYWORDS):
        tags.add('waterfall')
        tags.add('strenuous')

    if any(kw in combined for kw in BEACH_KEYWORDS):
        tags.add('beach')

    if any(kw in combined for kw in STRENUOUS_KEYWORDS):
        tags.add('strenuous')

    # if any(kw in combined for kw in HERITAGE_KEYWORDS):
    #     tags.add('heritage')

    return tags


def _is_must_see(poi) -> bool:
    """True if this POI qualifies as a must-see anchor."""
    return poi.normalized_popularity >= MUST_SEE_WPI_THRESHOLD


def _get_neighbours(poi, all_pois_in_route, radius_km: float = 5.0) -> list:
    """
    Return POIs in route that are within radius_km of the given POI.
    Used to reward clustering visits around must-see anchors.
    """
    neighbours = []
    for other in all_pois_in_route:
        if other is poi:
            continue
        dist = haversine_distance((poi.lat, poi.lon), (other.lat, other.lon))
        if dist <= radius_km:
            neighbours.append(other)
    return neighbours


# ============================================================
# ROUTE EVALUATION DATACLASS
# ============================================================

class RouteEvaluation:
    """Stores all components of a route evaluation."""
    def __init__(self):
        # ── Rewards ──
        self.poi_value_sum            = 0.0
        self.time_utilization_bonus   = 0.0
        self.route_length_bonus       = 0.0
        self.must_see_reward          = 0.0
        self.neighbour_reward         = 0.0
        self.correct_time_reward      = 0.0

        # ── Penalties ──
        self.distance_penalty         = 0.0
        self.hard_violation_penalty   = 0.0
        self.undertime_penalty        = 0.0
        self.overtime_penalty         = 0.0
        self.wrong_time_penalty       = 0.0
        self.waterfall_penalty        = 0.0
        self.fatigue_penalty          = 0.0

        # ── Tracking metrics ──
        self.total_travel_time_min    = 0.0
        self.total_visit_time_min     = 0.0
        self.total_distance_km        = 0.0
        self.closed_poi_count         = 0
        self.lunch_invasion_count     = 0
        self.overtime_minutes         = 0.0
        self.undertime_minutes        = 0.0
        self.total_time_min           = 0.0
        self.waterfall_count          = 0
        self.strenuous_count          = 0
        self.timeline                 = []

        # ── Combined scores ──
        self.delta                    = 0.0   # total penalty
        self.fitness                  = 0.0

        # ── Legacy compatibility ──
        self.travel_penalty           = 0.0
        self.constraint_penalty       = 0.0
        self.user_pref_penalty        = 0.0
        self.must_see_penalty         = 0.0
        self.restaurant_penalty       = 0.0


# ============================================================
# MAIN FITNESS FUNCTION
# ============================================================

def evaluate_fitness(route, start_time=TOUR_START_TIME):
    """
    Evaluate a tourism route with contextual intelligence.

    REWARDS:
      + POI value (WPI scores)
      + Time utilisation (filling 9am–6pm)
      + Route length (more stops = better up to budget)
      + Must-see POIs included
      + Neighbours of must-see POIs (geographic clustering)
      + Correct timing (beach at sunset, waterfall at dawn)

    PENALTIES:
      − Distance between stops
      − Closed POI visits
      − Lunch window invasions
      − Finishing too early (undertime)
      − Going over time budget (overtime)
      − Wrong-time visits (beach at 9am, waterfall at 5pm)
      − More than 1 waterfall per day
      − Too many physically demanding stops (fatigue)

    Formula:
        fitness = total_reward / (1 + delta)
    """
    eval_result = RouteEvaluation()

    if not route:
        eval_result.fitness = 0.0
        return eval_result

    current_time_min = time_to_minutes(start_time)
    lunch_start_min  = time_to_minutes(LUNCH_START_TIME)
    lunch_end_min    = time_to_minutes(LUNCH_END_TIME)

    # Pre-classify all POIs in this route
    poi_tags = {id(poi): _classify_poi(poi) for poi in route}

    # Count waterfalls and strenuous stops across whole route
    eval_result.waterfall_count  = sum(1 for poi in route if 'waterfall'  in poi_tags[id(poi)])
    eval_result.strenuous_count  = sum(1 for poi in route if 'strenuous'  in poi_tags[id(poi)])

    # ── 1. Simulate the tour ──────────────────────────────────────────────────
    for i, poi in enumerate(route):
        tags = poi_tags[id(poi)]

        # Travel from previous stop
        if i > 0:
            dist_km    = get_cached_distance(route[i - 1], poi)
            travel_min = calculate_travel_time(route[i - 1], poi)

            eval_result.total_distance_km     += dist_km
            eval_result.total_travel_time_min += travel_min
            eval_result.distance_penalty      += dist_km * DISTANCE_PENALTY_WEIGHT

            current_time_min += travel_min

        # Lunch window: auto-skip to end of lunch
        if lunch_start_min <= current_time_min < lunch_end_min:
            eval_result.lunch_invasion_count += 1
            eval_result.hard_violation_penalty += LUNCH_INVASION_PENALTY
            current_time_min = lunch_end_min

        arrival_time_min = current_time_min

        # Opening hours check
        poi_open_min  = time_to_minutes(poi.opening_time)
        poi_close_min = time_to_minutes(poi.closing_time)

        if current_time_min < poi_open_min:
            current_time_min = poi_open_min   # wait — no penalty

        elif current_time_min >= poi_close_min:
            eval_result.closed_poi_count += 1
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY

        visit_start_min = current_time_min
        visit_end_min   = current_time_min + poi.visit_duration_min

        if visit_end_min > poi_close_min and current_time_min < poi_close_min:
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY * 0.5

        eval_result.total_visit_time_min += poi.visit_duration_min
        current_time_min = visit_end_min

        # ── BEST-TIME SCHEDULING RULES ──────────────────────────────────────

        # WATERFALLS → should be in morning (before 11:00)
        # Reason: trekking is cooler, light is better, less crowded
        if 'waterfall' in tags:
            if visit_start_min <= time_to_minutes("11:00"):
                eval_result.correct_time_reward += CORRECT_TIME_REWARD
            elif visit_start_min >= AFTERNOON_END_MIN:
                # Very late waterfall — wrong time, trekking in evening heat
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY

        # BEACHES → best in late afternoon / evening (sunset)
        # Acceptable: morning (before 11am) also fine
        # Penalise: midday beach (12:00-15:00) — too hot
        if 'beach' in tags:
            if visit_start_min >= EVENING_START_MIN:
                # Evening / sunset beach — ideal
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 1.5
            elif visit_start_min <= time_to_minutes("11:00"):
                # Morning beach — acceptable
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 0.5
            elif MORNING_END_MIN <= visit_start_min < AFTERNOON_END_MIN:
                # Midday beach — hot and unpleasant
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY * 0.5

        # HERITAGE / CHURCHES / FORTS → morning or early afternoon
        # Reason: cooler, better lighting for photos, less crowded
        # if 'heritage' in tags:
        #     if visit_start_min <= time_to_minutes("13:00"):
        #         eval_result.correct_time_reward += CORRECT_TIME_REWARD * 0.5
        #     elif visit_start_min >= EVENING_START_MIN:
        #         # Heritage at dusk — many close by 5:30pm anyway
        #         eval_result.wrong_time_penalty += WRONG_TIME_PENALTY * 0.3
            
        

        eval_result.timeline.append((
            poi,
            minutes_to_time(int(arrival_time_min)),
            minutes_to_time(int(visit_start_min)),
            minutes_to_time(int(visit_end_min))
        ))
        
        

        # HARD: opening-hours feasibility
        poi_open_min = time_to_minutes(poi.opening_time)
        poi_close_min = time_to_minutes(poi.closing_time)

        if current_time_min < poi_open_min:
            current_time_min = poi_open_min  # wait allowed
        elif current_time_min >= poi_close_min:
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY
            eval_result.closed_poi_count += 1

    # ── 2. Time accounting ────────────────────────────────────────────────────
    eval_result.total_time_min = current_time_min - TOUR_START_MINUTES

    if current_time_min > TOUR_END_MINUTES:
        eval_result.overtime_minutes = current_time_min - TOUR_END_MINUTES
        eval_result.overtime_penalty = eval_result.overtime_minutes * OVERTIME_PENALTY_PER_MIN

    UNDERTIME_BUFFER_MIN = 60
    time_remaining = TOUR_END_MINUTES - current_time_min
    if time_remaining > UNDERTIME_BUFFER_MIN:
        eval_result.undertime_minutes  = time_remaining - UNDERTIME_BUFFER_MIN
        eval_result.undertime_penalty  = eval_result.undertime_minutes * UNDERTIME_PENALTY_PER_MIN
    
    # HARD: cannot exceed close time without penalty
    if visit_end_min > poi_close_min and visit_start_min < poi_close_min:
        eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY * 0.5

    # ── 3. Waterfall rule: max 1 per day ─────────────────────────────────────
    # Each waterfall beyond the first adds a heavy penalty.
    # Reason: waterfalls require trekking — doing 2 in one day is exhausting.
    if eval_result.waterfall_count > 1:
        extra = eval_result.waterfall_count - 1
        eval_result.waterfall_penalty = extra * EXTRA_WATERFALL_PENALTY

    # ── 4. Physical fatigue: max 2 strenuous stops per day ───────────────────
    # Strenuous = waterfalls + fort climbs + long treks
    if eval_result.strenuous_count > 2:
        extra = eval_result.strenuous_count - 2
        eval_result.fatigue_penalty = extra * PHYSICAL_FATIGUE_PENALTY

    # ── 5. Reward components ──────────────────────────────────────────────────

    # Base WPI reward
    eval_result.poi_value_sum = sum(poi.normalized_popularity for poi in route)

    # Time utilisation bonus
    time_used  = min(eval_result.total_time_min, DAILY_BUDGET_MIN)
    time_ratio = time_used / DAILY_BUDGET_MIN
    eval_result.time_utilization_bonus = time_ratio * TIME_UTILIZATION_WEIGHT

    # Route length bonus
    extra_pois = max(0, len(route) - MIN_POIS_PER_ROUTE)
    eval_result.route_length_bonus = extra_pois * ROUTE_LENGTH_BONUS_PER_POI

    # Must-see reward: bonus for including high-WPI anchor POIs
    must_see_pois = [p for p in route if _is_must_see(p)]
    eval_result.must_see_reward = len(must_see_pois) * MUST_SEE_REWARD

    # Neighbour reward: bonus for visiting POIs near must-see anchors
    # Encourages geographic clustering — visit what's nearby while you're there
    neighbour_bonus = 0.0
    for anchor in must_see_pois:
        neighbours_in_route = _get_neighbours(anchor, route, radius_km=5.0)
        neighbour_bonus += len(neighbours_in_route) * MUST_SEE_NEIGHBOUR_REWARD

    eval_result.neighbour_reward = neighbour_bonus

    best_start = time_to_minutes(getattr(poi, "best_time_start", "09:00"))
    best_end = time_to_minutes(getattr(poi, "best_time_end", "18:00"))

    if best_start <= visit_start_min <= best_end:
        eval_result.correct_time_reward += CORRECT_TIME_REWARD
    else:
        eval_result.wrong_time_penalty += WRONG_TIME_PENALTY * 0.4
    
    # Count strenuous from explicit intensity too
    is_strenuous = getattr(poi, "physical_intensity", "medium") == "high" or ("strenuous" in tags)

    crowd = float(getattr(poi, "crowd_factor", 0.5))
    eval_result.wrong_time_penalty += max(0.0, crowd - 0.7) * 2.0

    # ── 6. Assemble total penalty (delta) ─────────────────────────────────────
    eval_result.delta = (
        eval_result.distance_penalty
        + eval_result.hard_violation_penalty
        + eval_result.undertime_penalty
        + eval_result.overtime_penalty
        + eval_result.wrong_time_penalty
        + eval_result.waterfall_penalty
        + eval_result.fatigue_penalty
    )

    # ── 7. Final fitness ──────────────────────────────────────────────────────
    total_reward = (
        eval_result.poi_value_sum
        + eval_result.time_utilization_bonus
        + eval_result.route_length_bonus
        + eval_result.must_see_reward
        + eval_result.neighbour_reward
        + eval_result.correct_time_reward
    )

    eval_result.fitness = total_reward / (1.0 + eval_result.delta)

    # ── Legacy compatibility fields ───────────────────────────────────────────
    eval_result.travel_penalty     = eval_result.distance_penalty
    eval_result.constraint_penalty = eval_result.hard_violation_penalty
    eval_result.user_pref_penalty  = 0.0
    eval_result.must_see_penalty   = eval_result.waterfall_penalty + eval_result.fatigue_penalty
    eval_result.restaurant_penalty = 0.0

    return eval_result


# ── Quick config printout ─────────────────────────────────────────────────────
print('✅ Enhanced fitness function defined')
print(f'')
print(f'Tour window    : {TOUR_START_TIME} – {TOUR_END_TIME} ({DAILY_BUDGET_MIN} min)')
print(f'Must-see WPI   : ≥ {MUST_SEE_WPI_THRESHOLD}')
print(f'')
print(f'PENALTIES:')
print(f'  Distance      : {DISTANCE_PENALTY_WEIGHT}/km')
print(f'  Hard violation: {HARD_VIOLATION_PENALTY}/event')
print(f'  Undertime     : {UNDERTIME_PENALTY_PER_MIN}/min (60-min buffer)')
print(f'  Overtime      : {OVERTIME_PENALTY_PER_MIN}/min')
print(f'  Wrong time    : {WRONG_TIME_PENALTY}/event (beach noon, waterfall evening)')
print(f'  Extra waterfall: {EXTRA_WATERFALL_PENALTY} each beyond 1st')
print(f'  Fatigue       : {PHYSICAL_FATIGUE_PENALTY} each strenuous stop beyond 2')
print(f'')
print(f'REWARDS:')
print(f'  Time fill     : up to {TIME_UTILIZATION_WEIGHT}')
print(f'  Length bonus  : {ROUTE_LENGTH_BONUS_PER_POI}/POI beyond minimum')
print(f'  Must-see      : {MUST_SEE_REWARD}/anchor POI')
print(f'  Neighbours    : {MUST_SEE_NEIGHBOUR_REWARD}/neighbour of anchor')
print(f'  Correct time  : {CORRECT_TIME_REWARD}/event (beach sunset, waterfall morning)')

✅ Enhanced fitness function defined

Tour window    : 09:00 – 18:00 (540 min)
Must-see WPI   : ≥ 0.85

PENALTIES:
  Distance      : 0.5/km
  Hard violation: 50.0/event
  Undertime     : 0.15/min (60-min buffer)
  Overtime      : 0.2/min
  Wrong time    : 8.0/event (beach noon, waterfall evening)
  Extra waterfall: 30.0 each beyond 1st
  Fatigue       : 5.0 each strenuous stop beyond 2

REWARDS:
  Time fill     : up to 2.0
  Length bonus  : 0.05/POI beyond minimum
  Must-see      : 0.3/anchor POI
  Neighbours    : 0.15/neighbour of anchor
  Correct time  : 0.1/event (beach sunset, waterfall morning)


## 6. GA Operators

In [31]:
class Individual:
    """Represents a single route chromosome in the GA population."""
    def __init__(self, route, candidate_pois):
        self.route = route.copy() if isinstance(route, list) else list(route)
        self.candidate_pois = candidate_pois
        self.fitness = None
        self.evaluation = None
    
    def evaluate(self, start_time=TOUR_START_TIME):
        """Evaluate fitness of this individual's route."""
        self.evaluation = evaluate_fitness(self.route, start_time)
        self.fitness = self.evaluation.fitness
        return self.fitness
    
    def copy(self):
        """Create a deep copy of this individual."""
        new_ind = Individual(self.route, self.candidate_pois)
        new_ind.fitness = self.fitness
        new_ind.evaluation = self.evaluation
        return new_ind
    
    def __repr__(self):
        return f"Individual(route_len={len(self.route)}, fitness={self.fitness:.3f if self.fitness else 'None'})"

def initialize_population(candidate_pois, population_size):
    """Initialize population with random routes of varying lengths."""
    population = []
    
    for _ in range(population_size):
        # Random route length between MIN and MAX
        route_len = random.randint(MIN_POIS_PER_ROUTE, 
                                   min(MAX_POIS_PER_ROUTE, len(candidate_pois)))
        
        # Randomly sample POIs (prefer high-WPI POIs)
        # 70% chance to pick from top half by WPI, 30% from bottom half
        sorted_pois = sorted(candidate_pois, key=lambda p: p.normalized_popularity, reverse=True)
        mid = len(sorted_pois) // 2
        
        route = []
        for _ in range(route_len):
            if random.random() < 0.7 and mid > 0:
                # Pick from top half
                poi = random.choice(sorted_pois[:mid])
            else:
                # Pick from bottom half or all if mid==0
                poi = random.choice(sorted_pois)
            
            if poi not in route:
                route.append(poi)
        
        # Ensure minimum length
        while len(route) < MIN_POIS_PER_ROUTE:
            unvisited = [p for p in candidate_pois if p not in route]
            if unvisited:
                route.append(random.choice(unvisited))
            else:
                break
        
        population.append(Individual(route, candidate_pois))
    
    return population

def tournament_selection(population, tournament_size=TOURNAMENT_SIZE):
    """Select individual using tournament selection (TPOS spec: k=2)."""
    tournament = random.sample(population, tournament_size)
    return max(tournament, key=lambda ind: ind.fitness)

def swap_mutation(individual):
    """Swap mutation: randomly swap two POIs in the route (TPOS spec)."""
    if random.random() > MUTATION_RATE:
        return individual
    
    if len(individual.route) < 2:
        return individual
    
    # Swap two random positions
    mutated = individual.copy()
    idx1, idx2 = random.sample(range(len(mutated.route)), 2)
    mutated.route[idx1], mutated.route[idx2] = mutated.route[idx2], mutated.route[idx1]
    mutated.fitness = None  # Mark for re-evaluation
    
    return mutated

print('✅ Individual class and GA operators defined')


✅ Individual class and GA operators defined


In [32]:

def single_point_crossover(parent1, parent2):
    """
    Single-point crossover adapted for variable-length chromosomes (TPOS Section 2).
    
    - Crossover point chosen within minimum length of both parents
    - Duplicate genes rejected post-crossover (no location revisits)
    - Enforces MIN_POIS_PER_ROUTE to prevent route collapse
    - Probability: 0.7 (CROSSOVER_RATE)
    """
    size1 = len(parent1.route)
    size2 = len(parent2.route)
    
    # Crossover point within minimum length
    min_size = min(size1, size2)
    
    if min_size < 2:
        # Cannot crossover, return copies
        return parent1.copy(), parent2.copy()
    
    # Single crossover point within minimum length
    cx_point = random.randint(1, min_size - 1)
    
    # Create offspring
    offspring1_route = parent1.route[:cx_point].copy()
    offspring2_route = parent2.route[:cx_point].copy()
    
    # Add genes from other parent, avoiding duplicates
    for poi in parent2.route[cx_point:]:
        if poi not in offspring1_route:
            offspring1_route.append(poi)
    
    for poi in parent1.route[cx_point:]:
        if poi not in offspring2_route:
            offspring2_route.append(poi)
    
    # Enforce minimum route length
    # If crossover produced too-short route, add random unvisited POIs
    all_pois = parent1.candidate_pois
    
    while len(offspring1_route) < MIN_POIS_PER_ROUTE and all_pois:
        unvisited = [p for p in all_pois if p not in offspring1_route]
        if unvisited:
            # Prefer high-WPI POIs
            unvisited_sorted = sorted(unvisited, key=lambda p: p.normalized_popularity, reverse=True)
            new_poi = random.choice(unvisited_sorted[:min(5, len(unvisited_sorted))])
            offspring1_route.append(new_poi)
        else:
            break
    
    while len(offspring2_route) < MIN_POIS_PER_ROUTE and all_pois:
        unvisited = [p for p in all_pois if p not in offspring2_route]
        if unvisited:
            unvisited_sorted = sorted(unvisited, key=lambda p: p.normalized_popularity, reverse=True)
            new_poi = random.choice(unvisited_sorted[:min(5, len(unvisited_sorted))])
            offspring2_route.append(new_poi)
        else:
            break
    
    return Individual(offspring1_route, parent1.candidate_pois), Individual(offspring2_route, parent2.candidate_pois)



## 7. Main GA Loop

In [33]:
def optimize_route_ga(pois, start_time=TOUR_START_TIME):
    """
    Genetic Algorithm for route optimization (TPOS-aligned).

    Stops when either condition is met (TPOS Section 5):
    1. Fixed number of iterations reached (MAX_GENERATIONS)
    2. No significant fitness improvement (plateau detection)
    """

    # Safety checks
    if not pois or len(pois) == 0:
        raise ValueError("Cannot optimize route: No POIs provided")

    if len(pois) < MIN_POIS_PER_ROUTE:
        print(f"⚠️ Only {len(pois)} POIs available (min: {MIN_POIS_PER_ROUTE})")
        print(f"   Creating route with all available POIs")
        return Individual(pois, pois)

    print(f"\n🚀 Starting GA with {len(pois)} POIs")
    print(f"   Initial chromosome length: {INITIAL_CHROMOSOME_LENGTH} genes")

    # Initialize population
    population = initialize_population(pois, POPULATION_SIZE)

    # Evaluate initial population
    for ind in population:
        ind.evaluate(start_time)

    # Track best solution (elitism - TPOS Section 2)
    best_individual = max(population, key=lambda ind: ind.fitness)
    best_fitness_history = [best_individual.fitness]

    print(f"   Generation 0: Best fitness = {best_individual.fitness:.6f}")

    # Evolution loop
    no_improvement_count = 0

    for generation in range(1, MAX_GENERATIONS + 1):
        new_population = []

        # Elitism: preserve best solution (TPOS spec: 1 best)
        new_population.append(best_individual.copy())

        # Generate offspring
        while len(new_population) < POPULATION_SIZE:
            # Tournament selection (TPOS Section 2)
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            # Single-point crossover (TPOS Section 2)
            if random.random() < CROSSOVER_RATE:
                offspring1, offspring2 = single_point_crossover(parent1, parent2)
            else:
                offspring1, offspring2 = parent1.copy(), parent2.copy()

            # Swap mutation (TPOS Section 2)
            offspring1 = swap_mutation(offspring1)
            offspring2 = swap_mutation(offspring2)

            new_population.extend([offspring1, offspring2])

        # Trim to population size
        population = new_population[:POPULATION_SIZE]

        # Evaluate new population
        for ind in population:
            if ind.fitness is None:
                ind.evaluate(start_time)

        # Update best solution
        generation_best = max(population, key=lambda ind: ind.fitness)
        if generation_best.fitness > best_individual.fitness:
            best_individual = generation_best.copy()
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        best_fitness_history.append(best_individual.fitness)

        # Progress reporting (every 10 generations)
        if generation % 10 == 0:
            print(f"   Generation {generation}: Best fitness = {best_individual.fitness:.6f}, "
                  f"Route length = {len(best_individual.route)} POIs")

        # Early stopping (TPOS Section 5: plateau detection)
        if no_improvement_count >= EARLY_STOPPING_THRESHOLD:
            print(f"\n⏹️  Early stopping at generation {generation} "
                  f"(no improvement for {EARLY_STOPPING_THRESHOLD} generations)")
            break

    # Final results
    best_individual.evaluate(start_time)
    print(f"\n✅ GA completed:")
    print(f"   Final generation: {min(generation, MAX_GENERATIONS)}")
    print(f"   Best fitness: {best_individual.fitness:.6f}")
    print(f"   Route length: {len(best_individual.route)} POIs")
    print(f"   Total distance: {best_individual.evaluation.total_distance_km:.2f} km")
    print(f"   Total time: {best_individual.evaluation.total_time_min:.0f} min")
    print(f"   Delta (penalty sum): {best_individual.evaluation.delta:.2f}")

    return best_individual, best_fitness_history


print('✅ TPOS-aligned GA main loop defined')

✅ TPOS-aligned GA main loop defined


## 8. Run Optimization

**Development Tip:** Run the **single day test** first to verify everything works!

### 🧪 Test with ONE Day (Run This First!)

### 🔍 DEBUG: Detailed Logging (Run if stuck)

In [ ]:
# ========================================# DEBUG MODE: Detailed logging# ========================================TEST_DAY = 1print(f"\n{'='*60}")print(f"🔍 DEBUG: DAY {TEST_DAY} OPTIMIZATION")print('='*60)# Load POIsall_pois = load_pois_for_day(df, TEST_DAY)print(f"\n1️⃣ Loaded {len(all_pois)} POIs for Day {TEST_DAY}")selected_pois = select_top_pois_for_optimization(all_pois, max_candidates=50)print(f"\n2️⃣ All candidate POIs (GA will select subset):")for i, poi in enumerate(selected_pois, 1):    print(f"   {i}. {poi.name} (WPI: {poi.normalized_popularity:.3f})")# Build matricesprint(f"\n3️⃣ Building distance/time matrices...")prepare_day_matrices(selected_pois, use_osrm=True)# Test single route evaluationprint(f"\n4️⃣ Testing single route evaluation...")test_route = selected_pois.copy()print(f"   Route order: {[p.name[:20] for p in test_route]}")eval_result = evaluate_fitness(test_route)print(f"\n   📊 Fitness Breakdown:")print(f"      POI Value Sum: {eval_result.poi_value_sum:.3f}")print(f"      Travel Time: {eval_result.total_travel_time_min:.1f} min")print(f"      Visit Time: {eval_result.total_visit_time_min:.1f} min")print(f"      Total Time: {eval_result.total_time_min:.1f} min ({eval_result.total_time_min/60:.2f} hours)")print(f"      Distance: {eval_result.total_distance_km:.2f} km")print(f"      \n   ⚠️  Violations:")print(f"      Closed POIs: {eval_result.closed_poi_count}")print(f"      Lunch invasions: {eval_result.lunch_invasion_count}")print(f"      Overtime: {eval_result.overtime_minutes:.1f} min")print(f"      \n   💰 Penalties:")print(f"      Travel penalty: {eval_result.travel_penalty:.3f}")print(f"      Constraint penalty: {eval_result.constraint_penalty:.3f}")print(f"      \n   🎯 FINAL FITNESS: {eval_result.fitness:.3f}")if eval_result.fitness < 0:    print(f"\n   ❌ NEGATIVE FITNESS! Penalties are too high.")    print(f"      This means constraints are heavily violated.")    print(f"      POI value ({eval_result.poi_value_sum:.3f}) < Penalties ({eval_result.travel_penalty + eval_result.constraint_penalty:.3f})")# Show timelineprint(f"\n5️⃣ Timeline:")for i, (poi, arrival, visit_start, visit_end) in enumerate(eval_result.timeline, 1):    status = ""    if time_to_minutes(arrival) < time_to_minutes(poi.opening_time):        status = " ⚠️ EARLY (before opening)"    elif time_to_minutes(arrival) >= time_to_minutes(poi.closing_time):        status = " ❌ CLOSED"    print(f"   {i}. {arrival} - {poi.name[:30]} {status}")    print(f"      Open: {poi.opening_time}-{poi.closing_time} | Visit: {visit_start}-{visit_end}")# Now try GA with verbose loggingprint(f"\n6️⃣ Running GA with progress tracking...")print(f"   (This will show every 5 generations)\n")# Initialize populationpopulation = initialize_population(selected_pois, POPULATION_SIZE)print(f"   ✓ Population initialized: {len(population)} individuals")# Evaluate initial populationprint(f"   ✓ Evaluating initial population...")for ind in population:    ind.evaluate(TOUR_START_TIME)best_individual = max(population, key=lambda ind: ind.fitness)fitness_history = [best_individual.fitness]no_improvement_count = 0print(f"   ✓ Initial best fitness: {best_individual.fitness:.3f}")print(f"   ✓ Initial worst fitness: {min(population, key=lambda ind: ind.fitness).fitness:.3f}")print(f"   ✓ Initial avg fitness: {sum(ind.fitness for ind in population)/len(population):.3f}")# Run just 20 generations with verbose outputprint(f"\n   Running 20 generations (verbose)...\n")for generation in range(20):    # Selection and reproduction    offspring = []        while len(offspring) < POPULATION_SIZE:        parent1 = tournament_selection(population)        parent2 = tournament_selection(population)                if random.random() < CROSSOVER_RATE:            child1, child2 = single_point_crossover(parent1, parent2)        else:            child1, child2 = parent1.copy(), parent2.copy()                child1 = swap_mutation(child1)        child2 = swap_mutation(child2)                offspring.extend([child1, child2])        offspring = offspring[:POPULATION_SIZE]        # Evaluate offspring    for ind in offspring:        if ind.fitness is None:            ind.evaluate(TOUR_START_TIME)        # Elitism    population.sort(key=lambda ind: ind.fitness, reverse=True)    elites = population[:ELITE_COUNT]        offspring.sort(key=lambda ind: ind.fitness, reverse=True)    population = elites + offspring[:POPULATION_SIZE - ELITE_COUNT]        # Track best    generation_best = max(population, key=lambda ind: ind.fitness)    if generation_best.fitness > best_individual.fitness:        best_individual = generation_best        no_improvement_count = 0        improvement_marker = " ⬆️ IMPROVED"    else:        no_improvement_count += 1        improvement_marker = ""        fitness_history.append(best_individual.fitness)        if (generation + 1) % 5 == 0:        avg_fitness = sum(ind.fitness for ind in population) / len(population)        print(f"   Gen {generation+1:2d}: Best={best_individual.fitness:7.3f} | "              f"Avg={avg_fitness:7.3f} | No-improve={no_improvement_count}{improvement_marker}")        # Early stopping    if no_improvement_count >= EARLY_STOPPING_THRESHOLD:        print(f"\n   ⏹️  Early stopping at generation {generation+1}")        breakprint(f"\n✅ DEBUG complete!")print(f"\nFinal best fitness: {best_individual.fitness:.3f}")print(f"Fitness improved from {fitness_history[0]:.3f} to {fitness_history[-1]:.3f}")print(f"Improvement: {fitness_history[-1] - fitness_history[0]:.3f}")

In [34]:
# ========================================
# TEST: Optimize ONE day only
# ========================================

TEST_DAY = 1  # Change this to test different days

print(f"\n{'='*60}")
print(f"🧪 TEST: DAY {TEST_DAY} OPTIMIZATION")
print('='*60)

# Load all POIs and select top subset
all_pois = load_pois_for_day(df, TEST_DAY)
print(f"Loaded {len(all_pois)} POIs for Day {TEST_DAY}")

selected_pois = select_top_pois_for_optimization(all_pois)

# IMPORTANT: build distance/time matrices once for this day
prepare_day_matrices(selected_pois, use_osrm=True)

# Run GA optimization
best_individual, fitness_history = optimize_route_ga(selected_pois)

# Display results
print(f"\n{'='*60}")
print(f"DAY {TEST_DAY} RESULTS")
print('='*60)

print(f"\n📊 Fitness: {best_individual.fitness:.3f}")
print(f"   POI Value: {best_individual.evaluation.poi_value_sum:.3f}")
print(f"   Travel: {best_individual.evaluation.total_distance_km:.1f} km ({best_individual.evaluation.total_travel_time_min:.0f} min)")
print(f"   Time: {best_individual.evaluation.total_time_min/60:.2f} hours")

if best_individual.evaluation.closed_poi_count > 0 or best_individual.evaluation.lunch_invasion_count > 0:
    print(f"\n⚠️ Violations:")
    if best_individual.evaluation.closed_poi_count > 0:
        print(f"   Closed POIs: {best_individual.evaluation.closed_poi_count}")
    if best_individual.evaluation.lunch_invasion_count > 0:
        print(f"   Lunch invasions: {best_individual.evaluation.lunch_invasion_count}")

print(f"\n🗺️ Route ({len(best_individual.route)} POIs):")
for i, (poi, arrival, visit_start, visit_end) in enumerate(best_individual.evaluation.timeline, 1):
    print(f"   {i}. {poi.name}")
    print(f"      Arrive: {arrival} | Visit: {visit_start}-{visit_end} | WPI: {poi.normalized_popularity:.3f}")

print(f"\n✅ Single day test complete!")
print(f"\n💡 If this looks good, run the 'Full Multi-Day Optimization' cell below.")


🧪 TEST: DAY 1 OPTIMIZATION
Loaded 205 POIs for Day 1
   🚫 Removed 4 non-tourist venues (hotels, restaurants, etc.)
   🔧 Removed 51 near-duplicate POIs (similar names / same location)

   ✅ Final candidate pool: 50 POIs (from 205 original)
      WPI range : 0.696 – 1.000
      Types     : {'unknown': 49, 'default': 1}
✅ OSRM matrices ready: 50x50

🚀 Starting GA with 50 POIs
   Initial chromosome length: 6 genes
   Generation 0: Best fitness = 0.158129
   Generation 10: Best fitness = 0.258969, Route length = 6 POIs
   Generation 20: Best fitness = 0.303260, Route length = 6 POIs
   Generation 30: Best fitness = 0.303260, Route length = 6 POIs
   Generation 40: Best fitness = 0.303260, Route length = 6 POIs

⏹️  Early stopping at generation 40 (no improvement for 20 generations)

✅ GA completed:
   Final generation: 40
   Best fitness: 0.303260
   Route length: 6 POIs
   Total distance: 25.18 km
   Total time: 471 min
   Delta (penalty sum): 31.48

DAY 1 RESULTS

📊 Fitness: 0.303
   POI

### 🚀 Full Multi-Day Optimization (Run After Testing)

⚠️ **Only run this after verifying single day test works!**

This will optimize ALL days in your trip.

In [35]:
results = {}

for day in sorted(df["day"].unique()):
    print(f"\n{'='*60}")
    print(f"DAY {day} OPTIMIZATION")
    print("=" * 60)

    # Load all POIs and select top subset
    all_pois = load_pois_for_day(df, day)
    print(f"Loaded {len(all_pois)} POIs for Day {day}")

    selected_pois = select_top_pois_for_optimization(all_pois)

    # IMPORTANT: build distance/time matrices once for this day
    prepare_day_matrices(selected_pois, use_osrm=True)

    best_individual, fitness_history = optimize_route_ga(selected_pois)

    results[day] = {
        "best_individual": best_individual,
        "fitness_history": fitness_history,
        "selected_pois": selected_pois,
        "all_pois": all_pois,
        "selection_ratio": f"{len(selected_pois)}/{len(all_pois)}",
    }

print("\n✅ All days optimized!")


DAY 1 OPTIMIZATION
Loaded 205 POIs for Day 1
   🚫 Removed 4 non-tourist venues (hotels, restaurants, etc.)
   🔧 Removed 51 near-duplicate POIs (similar names / same location)

   ✅ Final candidate pool: 50 POIs (from 205 original)
      WPI range : 0.696 – 1.000
      Types     : {'unknown': 49, 'default': 1}
✅ OSRM matrices ready: 50x50

🚀 Starting GA with 50 POIs
   Initial chromosome length: 6 genes
   Generation 0: Best fitness = 0.189092
   Generation 10: Best fitness = 0.353267, Route length = 6 POIs
   Generation 20: Best fitness = 0.404420, Route length = 6 POIs
   Generation 30: Best fitness = 0.433176, Route length = 6 POIs
   Generation 40: Best fitness = 0.433176, Route length = 6 POIs

⏹️  Early stopping at generation 45 (no improvement for 20 generations)

✅ GA completed:
   Final generation: 45
   Best fitness: 0.433176
   Route length: 6 POIs
   Total distance: 11.88 km
   Total time: 455 min
   Delta (penalty sum): 23.26

DAY 2 OPTIMIZATION
Loaded 86 POIs for Day 2
  

## 9. Display Results

In [36]:
for day in sorted(results.keys()):
    print(f"\n{'='*60}")
    print(f"DAY {day} OPTIMIZED ROUTE")
    print('='*60)
    
    best = results[day]['best_individual']
    
    print(f"\n📊 Fitness: {best.fitness:.3f}")
    print(f"   POI Value: {best.evaluation.poi_value_sum:.3f}")
    print(f"   Travel: {best.evaluation.total_distance_km:.1f} km ({best.evaluation.total_travel_time_min:.0f} min)")
    print(f"   Time: {best.evaluation.total_time_min/60:.2f} hours")
    
    print(f"\n🗺️ Route:")
    for i, (poi, arrival, visit_start, visit_end) in enumerate(best.evaluation.timeline, 1):
        print(f"   {i}. {poi.name}")
        print(f"      Arrive: {arrival} | Visit: {visit_start}-{visit_end} | WPI: {poi.normalized_popularity:.3f}")


DAY 1 OPTIMIZED ROUTE

📊 Fitness: 0.433
   POI Value: 5.633
   Travel: 11.9 km (16 min)
   Time: 7.58 hours

🗺️ Route:
   1. BENZ WAX MUSEUM & FISH AQUARIUM
      Arrive: 09:00 | Visit: 09:00-10:00 | WPI: 0.983
   2. Candolim Beach
      Arrive: 10:06 | Visit: 10:06-11:06 | WPI: 0.983
   3. AAA All About Alcohol Museum
      Arrive: 11:08 | Visit: 11:08-12:08 | WPI: 0.884
   4. Sinquerim Fort
      Arrive: 13:30 | Visit: 13:30-14:30 | WPI: 0.952
   5. Dolphin Point
      Arrive: 14:33 | Visit: 14:33-15:33 | WPI: 0.863
   6. Fort Aguada
      Arrive: 15:34 | Visit: 15:34-16:34 | WPI: 0.969

DAY 2 OPTIMIZED ROUTE

📊 Fitness: 0.247
   POI Value: 4.247
   Travel: 7.0 km (13 min)
   Time: 6.58 hours

🗺️ Route:
   1. Patnem Beach
      Arrive: 09:00 | Visit: 09:00-10:00 | WPI: 0.901
   2. Backwaters Palolem
      Arrive: 10:05 | Visit: 10:05-11:05 | WPI: 0.830
   3. Palolem Beach
      Arrive: 11:07 | Visit: 11:07-12:07 | WPI: 1.000
   4. All day kayak adventure
      Arrive: 13:30 | Visit:

## 10. Visualize Routes

In [24]:
import folium
import requests
import polyline

# Use your existing config
OSRM_BASE_URL = "http://localhost:5000"

def get_osrm_segment_route(start_coord, end_coord, profile='driving'):
    """
    Get accurate route geometry for a single segment between two POIs.
    
    Args:
        start_coord: (lat, lon) tuple
        end_coord: (lat, lon) tuple
        profile: 'driving', 'walking', 'cycling'
    
    Returns:
        List of (lat, lon) tuples representing the actual road path
    """
    # OSRM expects lon,lat format
    coords_str = f"{start_coord[1]},{start_coord[0]};{end_coord[1]},{end_coord[0]}"
    url = f"{OSRM_BASE_URL}/route/v1/{profile}/{coords_str}"
    
    params = {
        'geometries': 'polyline6',  # High precision
        'overview': 'full',
        'steps': 'false',
        'alternatives': 'false',
        'continue_straight': 'false'  # Allow U-turns if needed
    }
    
    try:
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        
        if data.get('code') != 'Ok' or not data.get('routes'):
            # Fallback to straight line
            return [start_coord, end_coord]
        
        # Decode polyline6 to list of (lat, lon)
        geometry = data['routes'][0]['geometry']
        decoded = polyline.decode(geometry, precision=6)
        return decoded
        
    except Exception as e:
        # Silent fallback to straight line for this segment
        return [start_coord, end_coord]


def build_full_route_geometry(poi_coords, profile='driving'):
    """
    Build complete route geometry by concatenating consecutive segments.
    Avoids URL length limits and gives accurate turn-by-turn paths.
    
    Args:
        poi_coords: List of (lat, lon) tuples in visit order
    
    Returns:
        List of (lat, lon) tuples for the full route path
    """
    if len(poi_coords) < 2:
        return poi_coords
    
    full_geometry = []
    
    for i in range(len(poi_coords) - 1):
        start = poi_coords[i]
        end = poi_coords[i + 1]
        
        # Get road geometry for this segment
        segment = get_osrm_segment_route(start, end, profile)
        
        # Avoid duplicating the junction point (end of prev = start of next)
        if i > 0 and segment:
            segment = segment[1:]
        
        full_geometry.extend(segment)
    
    return full_geometry


def create_route_map(day, best_individual):
    """
    Create an interactive map with accurate OSRM routing between consecutive POIs.
    """
    route = best_individual.route
    timeline = best_individual.evaluation.timeline
    
    # Center map on route centroid
    center_lat = sum(poi.lat for poi in route) / len(route)
    center_lon = sum(poi.lon for poi in route) / len(route)
    m = folium.Map(
        location=[center_lat, center_lon], 
        zoom_start=11,
        tiles='CartoDB positron'  # Clean base map for route visibility
    )
    
    # Get POI coordinates in visit order
    poi_coords = [(poi.lat, poi.lon) for poi in route]
    
    # Build accurate route geometry segment by segment
    print(f"🛣️ Building OSRM route for Day {day} ({len(route)} stops)...")
    route_geometry = build_full_route_geometry(poi_coords)
    
    # Add the accurate route line
    folium.PolyLine(
        route_geometry,
        color='#2563eb',  # Nice blue
        weight=4,
        opacity=0.9,
        tooltip=f"Day {day} Driving Route",
        popup=f"Total stops: {len(route)}"
    ).add_to(m)
    
    # Optional: Add faint straight-line reference for comparison
    folium.PolyLine(
        poi_coords,
        color='#94a3b8',
        weight=2,
        opacity=0.4,
        dash_array='5, 10',
        tooltip="Direct distance (reference)"
    ).add_to(m)
    
    # Add numbered POI markers
    for i, (poi, arrival, visit_start, visit_end) in enumerate(timeline, 1):
        # Calculate travel info from previous POI
        travel_info = ""
        if i > 1:
            prev_poi = timeline[i-2][0]
            dist = get_cached_distance(prev_poi, poi)  # Use your existing helper
            time = calculate_travel_time(prev_poi, poi)  # Use your existing helper
            travel_info = f"<p><b>From previous:</b> {dist:.1f} km, {time:.0f} min</p>"
        
        popup_html = f"""
        <div style="font-family: system-ui, sans-serif; width: 260px; padding: 5px;">
            <h4 style="margin: 0 0 8px 0; color: #1e293b;">{i}. {poi.name}</h4>
            <div style="font-size: 13px; line-height: 1.4; color: #475569;">
                <p style="margin: 4px 0;"><b>⭐ Rating:</b> {poi.rating}/5 ({poi.user_ratings_total} reviews)</p>
                <p style="margin: 4px 0;"><b>📊 WPI Score:</b> {poi.normalized_popularity:.3f}</p>
                <hr style="border: none; border-top: 1px solid #e2e8f0; margin: 8px 0;">
                <p style="margin: 4px 0;"><b>🚗 Arrival:</b> {arrival}</p>
                <p style="margin: 4px 0;"><b>⏱️ Visit:</b> {visit_start} - {visit_end}</p>
                {travel_info}
            </div>
        </div>
        """
        
        # Color coding: Start=green, End=red, Middle=blue
        if i == 1:
            color = 'green'
            icon = 'play'
            prefix = 'fa'
        elif i == len(timeline):
            color = 'red'
            icon = 'flag-checkered'
            prefix = 'fa'
        else:
            color = 'blue'
            icon = 'star'
            prefix = 'fa'
        
        folium.Marker(
            [poi.lat, poi.lon],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Stop {i}: {poi.name}",
            icon=folium.Icon(
                color=color, 
                icon=icon, 
                prefix=prefix,
                icon_color='white'
            )
        ).add_to(m)
    
    # Add route summary legend
    total_distance = sum(
        get_cached_distance(route[i], route[i+1]) 
        for i in range(len(route)-1)
    )
    total_time = sum(
        calculate_travel_time(route[i], route[i+1]) 
        for i in range(len(route)-1)
    )
    
    legend_html = f'''
    <div style="position: fixed; bottom: 20px; right: 20px; 
                background: rgba(255,255,255,0.95); padding: 12px; 
                border-radius: 8px; border: 1px solid #cbd5e1; 
                font-family: system-ui; font-size: 13px; z-index: 9999;
                box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <b style="color: #1e293b; font-size: 14px;">Day {day} Summary</b><br>
        <div style="margin-top: 8px; color: #475569;">
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {len(route)} stops<br>
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {total_distance:.1f} km total<br>
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {total_time:.0f} min driving
        </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m


# Generate maps for all days
for day in sorted(results.keys()):
    best = results[day]['best_individual']
    route_map = create_route_map(day, best)
    filename = f'module4_route_day{day}_osrm.html'
    route_map.save(filename)
    print(f"✅ Saved: {filename}")

print("\n🗺️ All route maps generated with accurate OSRM geometry!")

🛣️ Building OSRM route for Day 1 (6 stops)...


KeyError: 131448864226656

## 11. Export Results to JSON

In [37]:
import json
import numpy as np

# Custom encoder to handle numpy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


output = {
    "num_days": len(results),
    "generated_at": str(pd.Timestamp.now()),
    "routes": {}
}

for day in sorted(results.keys()):
    best = results[day]['best_individual']

    route_data = []
    for i, (poi, arrival, visit_start, visit_end) in enumerate(best.evaluation.timeline, 1):
        route_data.append({
            "sequence": i,
            "poi_id": poi.place_id,
            "name": poi.name,
            "latitude": float(poi.lat),
            "longitude": float(poi.lon),
            "wpi_score": float(poi.normalized_popularity),
            "rating": float(poi.rating),
            "user_ratings_total": int(poi.user_ratings_total),
            "arrival_time": arrival,
            "visit_start": visit_start,
            "visit_end": visit_end,
            "visit_duration_min": int(poi.visit_duration_min)
        })

    output["routes"][f"day_{day}"] = {
        "day": int(day),
        "route": route_data,
        "fitness_score": float(best.fitness),
        "poi_value_sum": float(best.evaluation.poi_value_sum),
        "total_distance_km": float(best.evaluation.total_distance_km),
        "total_travel_time_min": float(best.evaluation.total_travel_time_min),
        "total_time_hours": float(best.evaluation.total_time_min / 60),
        "constraint_violations": {
            "closed_poi": int(best.evaluation.closed_poi_count),
            "lunch_invasion": int(best.evaluation.lunch_invasion_count),
            "overtime_min": float(best.evaluation.overtime_minutes)
        }
    }

with open('optimized_routes.json', 'w') as f:
    json.dump(output, f, indent=2, cls=NumpyEncoder)

print("✅ Saved: optimized_routes.json")
print(json.dumps(output, indent=2, cls=NumpyEncoder)[:1000] + "...")

✅ Saved: optimized_routes.json
{
  "num_days": 4,
  "generated_at": "2026-04-08 00:43:28.734196",
  "routes": {
    "day_1": {
      "day": 1,
      "route": [
        {
          "sequence": 1,
          "poi_id": "ChIJaQVTa_LrvzsRHDEKtziA-JE",
          "name": "BENZ WAX MUSEUM & FISH AQUARIUM",
          "latitude": 15.5609664,
          "longitude": 73.7636452,
          "wpi_score": 0.9829321244047392,
          "rating": 4.6,
          "user_ratings_total": 27315,
          "arrival_time": "09:00",
          "visit_start": "09:00",
          "visit_end": "10:00",
          "visit_duration_min": 60
        },
        {
          "sequence": 2,
          "poi_id": "ChIJv9F7IjzBvzsR70r6HmyML_o",
          "name": "Candolim Beach",
          "latitude": 15.5179749,
          "longitude": 73.7626122,
          "wpi_score": 0.9832669293960304,
          "rating": 4.5,
          "user_ratings_total": 42536,
          "arrival_time": "10:06",
          "visit_start": "10:06",
          "

In [38]:
import folium
import requests
import math
import json
import pandas as pd
from dataclasses import dataclass

OSRM_BASE_URL = "http://localhost:5000"

# ============================================
# Polyline decoder (no package needed)
# ============================================

def decode_polyline(encoded, precision=6):
    """Pure Python polyline decoder."""
    if not encoded:
        return []
    
    coords = []
    index = 0
    lat = 0
    lon = 0
    factor = math.pow(10, precision)

    while index < len(encoded):
        shift = 0
        result = 0
        while True:
            byte = ord(encoded[index]) - 63
            index += 1
            result |= (byte & 0x1f) << shift
            shift += 5
            if byte < 0x20:
                break
        
        lat += ~(result >> 1) if (result & 1) else (result >> 1)

        shift = 0
        result = 0
        while True:
            byte = ord(encoded[index]) - 63
            index += 1
            result |= (byte & 0x1f) << shift
            shift += 5
            if byte < 0x20:
                break
        
        lon += ~(result >> 1) if (result & 1) else (result >> 1)

        coords.append((lat / factor, lon / factor))
    
    return coords


# ============================================
# OSRM routing functions
# ============================================

def get_osrm_segment_route(start_coord, end_coord, profile='driving'):
    """Get accurate route geometry for a single segment."""
    coords_str = f"{start_coord[1]},{start_coord[0]};{end_coord[1]},{end_coord[0]}"
    url = f"{OSRM_BASE_URL}/route/v1/{profile}/{coords_str}"
    
    params = {
        'geometries': 'polyline6',
        'overview': 'full',
        'steps': 'false',
        'alternatives': 'false',
        'continue_straight': 'false'
    }
    
    try:
        resp = requests.get(url, params=params, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        
        if data.get('code') != 'Ok' or not data.get('routes'):
            return [start_coord, end_coord]
        
        geometry = data['routes'][0]['geometry']
        return decode_polyline(geometry, precision=6)
        
    except Exception as e:
        return [start_coord, end_coord]


def build_full_route_geometry(coords_list, profile='driving'):
    """Build complete route by concatenating consecutive segments."""
    if len(coords_list) < 2:
        return coords_list
    
    full_geometry = []
    
    for i in range(len(coords_list) - 1):
        start = coords_list[i]
        end = coords_list[i + 1]
        
        segment = get_osrm_segment_route(start, end, profile)
        
        if i > 0 and segment:
            segment = segment[1:]
        
        full_geometry.extend(segment)
    
    return full_geometry


# ============================================
# Map creation from JSON
# ============================================

def create_route_map_from_json(day_key, route_data):
    """
    Create map directly from JSON route data.
    No dependency on POI objects or matrix cache!
    """
    stops = route_data['route']
    
    if not stops:
        return None
    
    # Center on route
    center_lat = sum(s['latitude'] for s in stops) / len(stops)
    center_lon = sum(s['longitude'] for s in stops) / len(stops)
    
    m = folium.Map(
        location=[center_lat, center_lon], 
        zoom_start=11,
        tiles='CartoDB positron'
    )
    
    # Extract coordinates
    coords = [(s['latitude'], s['longitude']) for s in stops]
    
    # Build OSRM route
    print(f"🛣️ Building OSRM route for {day_key} ({len(stops)} stops)...")
    route_geometry = build_full_route_geometry(coords)
    
    # Accurate road route
    folium.PolyLine(
        route_geometry,
        color='#2563eb',
        weight=4,
        opacity=0.9,
        tooltip=f"{day_key} Driving Route"
    ).add_to(m)
    
    # Reference straight lines
    folium.PolyLine(
        coords,
        color='#94a3b8',
        weight=2,
        opacity=0.4,
        dash_array='5, 10',
        tooltip="Direct distance"
    ).add_to(m)
    
    # Add markers with travel info from JSON
    for i, stop in enumerate(stops, 1):
        # Calculate travel info from previous stop
        travel_info = ""
        if i > 1:
            prev = stops[i-2]
            # Use OSRM for segment distance/time or calculate from JSON totals
            # For now, we'll fetch fresh or use haversine as fallback
            dist = haversine_distance(
                (prev['latitude'], prev['longitude']),
                (stop['latitude'], stop['longitude'])
            )
            travel_info = f"<p><b>From previous:</b> ~{dist:.1f} km</p>"
        
        popup_html = f"""
        <div style="font-family: system-ui, sans-serif; width: 260px; padding: 5px;">
            <h4 style="margin: 0 0 8px 0; color: #1e293b;">{stop['sequence']}. {stop['name']}</h4>
            <div style="font-size: 13px; line-height: 1.4; color: #475569;">
                <p style="margin: 4px 0;"><b>⭐ Rating:</b> {stop['rating']}/5 ({stop['user_ratings_total']} reviews)</p>
                <p style="margin: 4px 0;"><b>📊 WPI Score:</b> {stop['wpi_score']:.3f}</p>
                <hr style="border: none; border-top: 1px solid #e2e8f0; margin: 8px 0;">
                <p style="margin: 4px 0;"><b>🚗 Arrival:</b> {stop['arrival_time']}</p>
                <p style="margin: 4px 0;"><b>⏱️ Visit:</b> {stop['visit_start']} - {stop['visit_end']}</p>
                <p style="margin: 4px 0;"><b>Duration:</b> {stop['visit_duration_min']} min</p>
                {travel_info}
            </div>
        </div>
        """
        
        if i == 1:
            color, icon = 'green', 'play'
        elif i == len(stops):
            color, icon = 'red', 'flag-checkered'
        else:
            color, icon = 'blue', 'star'
        
        folium.Marker(
            [stop['latitude'], stop['longitude']],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Stop {stop['sequence']}: {stop['name']}",
            icon=folium.Icon(color=color, icon=icon, prefix='fa', icon_color='white')
        ).add_to(m)
    
    # Add summary
    summary = route_data
    legend_html = f'''
    <div style="position: fixed; bottom: 20px; right: 20px; 
                background: rgba(255,255,255,0.95); padding: 12px; 
                border-radius: 8px; border: 1px solid #cbd5e1; 
                font-family: system-ui; font-size: 13px; z-index: 9999;
                box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <b style="color: #1e293b; font-size: 14px;">{day_key} Summary</b><br>
        <div style="margin-top: 8px; color: #475569;">
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {len(stops)} stops<br>
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {summary['total_distance_km']:.1f} km total<br>
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            {summary['total_travel_time_min']:.0f} min driving<br>
            <span style="display: inline-block; width: 8px; height: 8px; 
                   background: #2563eb; border-radius: 50%; margin-right: 6px;"></span>
            Fitness: {summary['fitness_score']:.3f}
        </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m


def haversine_distance(coord1, coord2):
    """Calculate great-circle distance between two points."""
    lat1, lon1 = map(math.radians, coord1)
    lat2, lon2 = map(math.radians, coord2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return 6371.0 * c


# ============================================
# MAIN: Load JSON and create maps
# ============================================

# Load your saved JSON
with open('optimized_routes.json', 'r') as f:
    data = json.load(f)

# Create maps for each day
for day_key, route_data in data['routes'].items():
    route_map = create_route_map_from_json(day_key, route_data)
    if route_map:
        filename = f'module4_{day_key}_osrm_map.html'
        route_map.save(filename)
        print(f"✅ Saved: {filename}")

print("\n🗺️ All maps created from JSON data!")

🛣️ Building OSRM route for day_1 (6 stops)...
✅ Saved: module4_day_1_osrm_map.html
🛣️ Building OSRM route for day_2 (5 stops)...
✅ Saved: module4_day_2_osrm_map.html
🛣️ Building OSRM route for day_3 (6 stops)...
✅ Saved: module4_day_3_osrm_map.html
🛣️ Building OSRM route for day_4 (7 stops)...
✅ Saved: module4_day_4_osrm_map.html

🗺️ All maps created from JSON data!


In [19]:
import json
import math
import requests
from datetime import datetime
from typing import List, Dict, Any

# ============================================
# CONFIGURATION
# ============================================

OSRM_BASE_URL = "http://localhost:5000"  # Your local OSRM

# ============================================
# POLYLINE DECODER (for OSRM)
# ============================================

def decode_polyline(encoded: str, precision: int = 6) -> List[List[float]]:
    """Decode OSRM polyline6 to [[lon, lat], ...] for GeoJSON."""
    if not encoded:
        return []
    
    coords = []
    index = 0
    lat = 0
    lon = 0
    factor = math.pow(10, precision)

    while index < len(encoded):
        shift = 0
        result = 0
        while True:
            byte = ord(encoded[index]) - 63
            index += 1
            result |= (byte & 0x1f) << shift
            shift += 5
            if byte < 0x20:
                break
        
        lat += ~(result >> 1) if (result & 1) else (result >> 1)

        shift = 0
        result = 0
        while True:
            byte = ord(encoded[index]) - 63
            index += 1
            result |= (byte & 0x1f) << shift
            shift += 5
            if byte < 0x20:
                break
        
        lon += ~(result >> 1) if (result & 1) else (result >> 1)

        coords.append([lon / factor, lat / factor])  # GeoJSON: [lon, lat]
    
    return coords


# ============================================
# OSRM SEGMENT FETCHER
# ============================================

def get_osrm_segment(start_lat: float, start_lon: float, 
                     end_lat: float, end_lon: float) -> Dict[str, Any]:
    """Fetch route from OSRM for one segment."""
    coords_str = f"{start_lon},{start_lat};{end_lon},{end_lat}"
    url = f"{OSRM_BASE_URL}/route/v1/driving/{coords_str}"
    
    try:
        resp = requests.get(url, params={
            'geometries': 'polyline6',
            'overview': 'full',
            'steps': 'false',
            'alternatives': 'false',
            'continue_straight': 'false'
        }, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        
        if data.get('code') != 'Ok' or not data.get('routes'):
            return None
        
        route = data['routes'][0]
        return {
            'geometry': decode_polyline(route['geometry'], precision=6),
            'distance_km': round(route['distance'] / 1000, 2),
            'duration_min': round(route['duration'] / 60, 1)
        }
    except Exception as e:
        print(f"  ⚠️  OSRM failed for segment: {e}")
        return None


def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate straight-line distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return round(R * c, 2)


# ============================================
# DAY JSON BUILDER
# ============================================

def build_day_json(day_num: int, best_individual) -> Dict[str, Any]:
    """
    Build mobile-optimized JSON for a single day.
    """
    route = best_individual.route
    timeline = best_individual.evaluation.timeline
    
    print(f"🗺️  Building Day {day_num} ({len(route)} stops)...")
    
    day_data = {
        "meta": {
            "version": "2.0",
            "day": int(day_num),
            "generated_at": datetime.now().isoformat(),
            "profile": "driving",
            "stops": len(route)
        },
        "stats": {
            "total_distance_km": float(best_individual.evaluation.total_distance_km),
            "total_drive_min": float(best_individual.evaluation.total_travel_time_min),
            "start_time": timeline[0][2] if timeline else None,
            "end_time": timeline[-1][3] if timeline else None,
            "fitness_score": float(best_individual.fitness)
        },
        "violations": {
            "closed_poi": int(best_individual.evaluation.closed_poi_count),
            "lunch_invasion": int(best_individual.evaluation.lunch_invasion_count),
            "overtime_min": float(best_individual.evaluation.overtime_minutes)
        },
        "stops": [],
        "legs": []
    }
    
    # Build stops array
    for i, (poi, arrival, visit_start, visit_end) in enumerate(timeline, 1):
        stop = {
            "i": i,
            "id": poi.place_id,
            "name": poi.name,
            "lat": round(float(poi.lat), 6),
            "lon": round(float(poi.lon), 6),
            "wpi": round(float(poi.normalized_popularity), 3),
            "rating": float(poi.rating),
            "reviews": int(poi.user_ratings_total),
            "arrive": arrival,
            "visit_start": visit_start,
            "visit_end": visit_end,
            "stay_min": int(poi.visit_duration_min),
            "type": "start" if i == 1 else ("end" if i == len(timeline) else "stop")
        }
        day_data["stops"].append(stop)
    
    # Build legs (OSRM routes between stops)
    total_osrm_distance = 0
    total_osrm_duration = 0
    
    for i in range(len(timeline) - 1):
        current = timeline[i][0]
        next_poi = timeline[i + 1][0]
        
        print(f"  📍 Routing: Stop {i+1} → Stop {i+2} ({current.name[:20]}... → {next_poi.name[:20]}...)")
        
        # Get OSRM route
        osrm_data = get_osrm_segment(
            current.lat, current.lon,
            next_poi.lat, next_poi.lon
        )
        
        if osrm_data:
            # Encode geometry back to polyline6 for compact storage
            # (Frontend will decode this)
            # Note: We store the OSRM-returned polyline string, not decoded coords
            # So we need to re-encode or store the original string
            
            # Actually, let's store the original polyline string from OSRM
            # But we decoded it above, so let's fetch again or store differently
            
            # Better approach: store the decoded coordinates as compact array
            # or re-fetch and store the polyline string
            
            # For now, let's store decoded coords as compact array (more reliable)
            leg = {
                "from_i": i + 1,
                "to_i": i + 2,
                "geometry": osrm_data['geometry'],  # [[lon, lat], [lon, lat], ...]
                "km": osrm_data['distance_km'],
                "min": osrm_data['duration_min']
            }
            total_osrm_distance += osrm_data['distance_km']
            total_osrm_duration += osrm_data['duration_min']
        else:
            # Fallback to straight line
            straight_distance = haversine(
                current.lat, current.lon,
                next_poi.lat, next_poi.lon
            )
            leg = {
                "from_i": i + 1,
                "to_i": i + 2,
                "geometry": None,  # Frontend draws straight line
                "km": straight_distance,
                "min": round((straight_distance / 30.0) * 60, 1),  # Assume 30km/h
                "fallback": True
            }
        
        day_data["legs"].append(leg)
    
    # Update stats with actual OSRM totals
    day_data["stats"]["actual_distance_km"] = round(total_osrm_distance, 1)
    day_data["stats"]["actual_drive_min"] = round(total_osrm_duration, 1)
    
    print(f"  ✅ Day {day_num} complete: {total_osrm_distance:.1f} km, {total_osrm_duration:.0f} min driving")
    
    return day_data


# ============================================
# SAVE TO FILES
# ============================================

def save_days_separately(results: Dict, trip_name: str = "trip"):
    """
    Save each day as a separate compact JSON file.
    Also creates a summary file.
    
    Returns: List of filenames created
    """
    created_files = []
    trip_id = f"{trip_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    print(f"\n💾 Saving trip: {trip_id}")
    print("=" * 50)
    
    # Process each day
    day_summaries = []
    
    for day_num in sorted(results.keys()):
        best = results[day_num]['best_individual']
        
        # Build JSON for this day
        day_json = build_day_json(day_num, best)
        
        # Save to file (compact format)
        filename = f"day_{day_num}.json"
        with open(filename, 'w') as f:
            json.dump(day_json, f, separators=(',', ':'))  # Compact, no spaces
        
        # Get file size
        file_size = len(json.dumps(day_json, separators=(',', ':')))
        
        print(f"💾 Saved: {filename} ({file_size/1024:.1f} KB)")
        created_files.append(filename)
        
        # Build summary entry
        day_summaries.append({
            "day": int(day_num),
            "file": filename,
            "stops": day_json["meta"]["stops"],
            "km": day_json["stats"]["actual_distance_km"],
            "drive_min": day_json["stats"]["actual_drive_min"],
            "start": day_json["stats"]["start_time"],
            "end": day_json["stats"]["end_time"]
        })
    
    # Create summary file
    summary = {
        "trip_id": trip_id,
        "created_at": datetime.now().isoformat(),
        "total_days": len(results),
        "days": day_summaries
    }
    
    summary_filename = "trip_summary.json"
    with open(summary_filename, 'w') as f:
        json.dump(summary, f, indent=2)  # Formatted for readability
    
    print(f"📋 Summary saved: {summary_filename}")
    created_files.append(summary_filename)
    
    print("\n" + "=" * 50)
    print(f"✅ All files saved! Total: {len(created_files)} files")
    print(f"   - {len(results)} day files (compact)")
    print(f"   - 1 summary file (formatted)")
    
    return created_files, trip_id


# ============================================
# USAGE IN YOUR NOTEBOOK
# ============================================

# After your GA completes and you have 'results' dict:
# files, trip_id = save_days_separately(results, trip_name="goa_trip")

# Example of what this creates:
print("\nExample output structure:")
print("""
📁 Working Directory/
   ├── day_1.json          (15 KB, compact)
   ├── day_2.json          (18 KB, compact)  
   ├── day_3.json          (12 KB, compact)
   └── trip_summary.json   (2 KB, formatted)
""")

# To load and verify:
print("\nTo load a specific day in notebook:")
print("""
with open('day_1.json', 'r') as f:
    day_1_data = json.load(f)
    
print(day_1_data['meta']['day'])        # 1
print(day_1_data['stats']['km'])        # 45.2
print(len(day_1_data['stops']))         # 5
print(len(day_1_data['legs']))          # 4
""")


Example output structure:

📁 Working Directory/
   ├── day_1.json          (15 KB, compact)
   ├── day_2.json          (18 KB, compact)  
   ├── day_3.json          (12 KB, compact)
   └── trip_summary.json   (2 KB, formatted)


To load a specific day in notebook:

with open('day_1.json', 'r') as f:
    day_1_data = json.load(f)
    
print(day_1_data['meta']['day'])        # 1
print(day_1_data['stats']['km'])        # 45.2
print(len(day_1_data['stops']))         # 5
print(len(day_1_data['legs']))          # 4



In [33]:
def verify_saved_files():
    """Quick verification that files are valid."""
    print("\n🔍 Verifying saved files...")
    
    # Check summary
    with open('trip_summary.json', 'r') as f:
        summary = json.load(f)
    
    print(f"Trip: {summary['trip_id']}")
    print(f"Days: {summary['total_days']}")
    
    # Check each day
    for day_info in summary['days']:
        filename = day_info['file']
        with open(filename, 'r') as f:
            data = json.load(f)
        
        # Verify structure
        assert 'meta' in data
        assert 'stops' in data
        assert 'legs' in data
        assert len(data['legs']) == len(data['stops']) - 1
        
        print(f"  ✅ {filename}: {len(data['stops'])} stops, {len(data['legs'])} legs")
    
    print("✅ All files valid!")

# Run after saving:
files, trip_id = save_days_separately(results)
verify_saved_files()


💾 Saving trip: trip_20260407_001859
🗺️  Building Day 1 (6 stops)...
  📍 Routing: Stop 1 → Stop 2 (Patnem Beach... → Sahil Boat Trips...)
  📍 Routing: Stop 2 → Stop 3 (Sahil Boat Trips... → Palolem Beach, GOA...)
  📍 Routing: Stop 3 → Stop 4 (Palolem Beach, GOA... → Palolem Beach...)
  📍 Routing: Stop 4 → Stop 5 (Palolem Beach... → Backwaters Palolem...)
  📍 Routing: Stop 5 → Stop 6 (Backwaters Palolem... → Butterfly Beach Goa...)
  ✅ Day 1 complete: 12.8 km, 26 min driving
💾 Saved: day_1.json (9.1 KB)
🗺️  Building Day 2 (5 stops)...
  📍 Routing: Stop 1 → Stop 2 (Se Cathedral... → Church of St. Franci...)
  📍 Routing: Stop 2 → Stop 3 (Church of St. Franci... → Archaeological Surve...)
  📍 Routing: Stop 3 → Stop 4 (Archaeological Surve... → Basilica of Bom Jesu...)
  📍 Routing: Stop 4 → Stop 5 (Basilica of Bom Jesu... → St. Augustine Tower...)
  ✅ Day 2 complete: 1.4 km, 4 min driving
💾 Saved: day_2.json (2.8 KB)
🗺️  Building Day 3 (4 stops)...
  📍 Routing: Stop 1 → Stop 2 (Chorla Ghat.

## ✅ Module 4 Complete!

In [30]:
# Improved fitness override (v2)
# Run this cell after existing fitness definitions.

BACKTRACK_PENALTY_WEIGHT = 0.25
DISTANCE_QUADRATIC_WEIGHT = 0.015
OVERTIME_HARD_THRESHOLD_MIN = 60
OVERTIME_STEEP_MULTIPLIER = 2.0


def evaluate_fitness(route, start_time=TOUR_START_TIME):
    """
    Improved fitness focused on itinerary quality and stability.
    Keeps compatibility with RouteEvaluation fields used elsewhere.
    """
    eval_result = RouteEvaluation()

    if not route:
        eval_result.fitness = 0.0
        return eval_result

    current_time_min = time_to_minutes(start_time)
    lunch_start_min = time_to_minutes(LUNCH_START_TIME)
    lunch_end_min = time_to_minutes(LUNCH_END_TIME)

    # Tag all POIs once
    poi_tags = {id(poi): _classify_poi(poi) for poi in route}
    eval_result.waterfall_count = sum(1 for poi in route if 'waterfall' in poi_tags[id(poi)])
    eval_result.strenuous_count = sum(1 for poi in route if 'strenuous' in poi_tags[id(poi)])

    # Track for smoothness penalty
    segment_distances = []

    for i, poi in enumerate(route):
        tags = poi_tags[id(poi)]

        if i > 0:
            prev = route[i - 1]
            dist_km = get_cached_distance(prev, poi)
            travel_min = calculate_travel_time(prev, poi)

            eval_result.total_distance_km += dist_km
            eval_result.total_travel_time_min += travel_min

            # Nonlinear distance penalty: discourages long jumps strongly
            eval_result.distance_penalty += (
                DISTANCE_PENALTY_WEIGHT * dist_km
                + DISTANCE_QUADRATIC_WEIGHT * (dist_km ** 2)
            )
            segment_distances.append(dist_km)
            current_time_min += travel_min

        if lunch_start_min <= current_time_min < lunch_end_min:
            eval_result.lunch_invasion_count += 1
            eval_result.hard_violation_penalty += LUNCH_INVASION_PENALTY
            current_time_min = lunch_end_min

        arrival_time_min = current_time_min

        poi_open_min = time_to_minutes(poi.opening_time)
        poi_close_min = time_to_minutes(poi.closing_time)

        if current_time_min < poi_open_min:
            current_time_min = poi_open_min
        elif current_time_min >= poi_close_min:
            eval_result.closed_poi_count += 1
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY

        visit_start_min = current_time_min
        visit_end_min = current_time_min + poi.visit_duration_min

        if visit_end_min > poi_close_min and current_time_min < poi_close_min:
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY * 0.5

        eval_result.total_visit_time_min += poi.visit_duration_min
        current_time_min = visit_end_min

        # Timing rewards/penalties
        if 'waterfall' in tags:
            if visit_start_min <= time_to_minutes('11:00'):
                eval_result.correct_time_reward += CORRECT_TIME_REWARD
            elif visit_start_min >= AFTERNOON_END_MIN:
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY

        if 'beach' in tags:
            if visit_start_min >= EVENING_START_MIN:
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 1.5
            elif visit_start_min <= time_to_minutes('11:00'):
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 0.5
            elif MORNING_END_MIN <= visit_start_min < AFTERNOON_END_MIN:
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY * 0.5

        eval_result.timeline.append((
            poi,
            minutes_to_time(int(arrival_time_min)),
            minutes_to_time(int(visit_start_min)),
            minutes_to_time(int(visit_end_min))
        ))

    # Day-level penalties/rewards
    eval_result.total_time_min = current_time_min - TOUR_START_MINUTES

    if current_time_min > TOUR_END_MINUTES:
        eval_result.overtime_minutes = current_time_min - TOUR_END_MINUTES
        base_over = eval_result.overtime_minutes * OVERTIME_PENALTY_PER_MIN
        if eval_result.overtime_minutes > OVERTIME_HARD_THRESHOLD_MIN:
            base_over *= OVERTIME_STEEP_MULTIPLIER
        eval_result.overtime_penalty = base_over

    undertime_buffer_min = 45
    remaining = TOUR_END_MINUTES - current_time_min
    if remaining > undertime_buffer_min:
        eval_result.undertime_minutes = remaining - undertime_buffer_min
        eval_result.undertime_penalty = eval_result.undertime_minutes * UNDERTIME_PENALTY_PER_MIN

    if eval_result.waterfall_count > 1:
        eval_result.waterfall_penalty = (eval_result.waterfall_count - 1) * EXTRA_WATERFALL_PENALTY

    if eval_result.strenuous_count > 2:
        eval_result.fatigue_penalty = (eval_result.strenuous_count - 2) * PHYSICAL_FATIGUE_PENALTY

    # Backtracking / route smoothness proxy
    if len(segment_distances) >= 2:
        avg_d = sum(segment_distances) / len(segment_distances)
        peaks = sum(max(0.0, d - avg_d) for d in segment_distances)
        eval_result.wrong_time_penalty += peaks * BACKTRACK_PENALTY_WEIGHT

    # Rewards
    eval_result.poi_value_sum = sum(p.normalized_popularity for p in route)

    time_used = min(eval_result.total_time_min, DAILY_BUDGET_MIN)
    time_ratio = time_used / DAILY_BUDGET_MIN if DAILY_BUDGET_MIN else 0.0
    eval_result.time_utilization_bonus = time_ratio * TIME_UTILIZATION_WEIGHT

    # Target route length bonus around practical count
    target_pois = 8 if DAILY_BUDGET_MIN >= 480 else 6
    length_gap = abs(len(route) - target_pois)
    eval_result.route_length_bonus = max(0.0, 0.6 - 0.08 * length_gap)

    must_see_pois = [p for p in route if _is_must_see(p)]
    eval_result.must_see_reward = len(must_see_pois) * MUST_SEE_REWARD

    # Unique neighbour reward to avoid double counting
    neighbour_ids = set()
    for anchor in must_see_pois:
        for n in _get_neighbours(anchor, route, radius_km=5.0):
            neighbour_ids.add(id(n))
    eval_result.neighbour_reward = len(neighbour_ids) * MUST_SEE_NEIGHBOUR_REWARD

    eval_result.delta = (
        eval_result.distance_penalty
        + eval_result.hard_violation_penalty
        + eval_result.undertime_penalty
        + eval_result.overtime_penalty
        + eval_result.wrong_time_penalty
        + eval_result.waterfall_penalty
        + eval_result.fatigue_penalty
    )

    total_reward = (
        eval_result.poi_value_sum
        + eval_result.time_utilization_bonus
        + eval_result.route_length_bonus
        + eval_result.must_see_reward
        + eval_result.neighbour_reward
        + eval_result.correct_time_reward
    )

    eval_result.fitness = total_reward / (1.0 + eval_result.delta)

    # Legacy fields
    eval_result.travel_penalty = eval_result.distance_penalty
    eval_result.constraint_penalty = eval_result.hard_violation_penalty
    eval_result.user_pref_penalty = 0.0
    eval_result.must_see_penalty = eval_result.waterfall_penalty + eval_result.fatigue_penalty
    eval_result.restaurant_penalty = 0.0

    return eval_result

print('✅ Improved evaluate_fitness() override loaded')

✅ Improved evaluate_fitness() override loaded


In [ ]:
# Improved fitness override (v3): nearby-500m + park timing support
# Run this AFTER previous fitness cells.

PARK_KEYWORDS = {
    'park', 'garden', 'botanical', 'eco park', 'national park',
    'wildlife sanctuary', 'nature park', 'bird park',
    'amusement park', 'theme park', 'water park', 'adventure park',
    'outdoor park', 'recreation park', 'playground'
}

PARK_TIME_START_MIN = time_to_minutes('12:00')
PARK_TIME_END_MIN = time_to_minutes('16:00')
PARK_WRONG_TIME_PENALTY = 6.0
PARK_CORRECT_TIME_REWARD = 0.08

NEARBY_DEDUP_KM = 0.5
NEARBY_REWARD_RADIUS_KM = 0.7
NEARBY_DUPLICATE_PENALTY = 4.0


def _classify_poi(poi) -> set:
    tags = set()
    name = poi.name.lower()
    ptype = (getattr(poi, 'type', '') or '').lower()
    combined = name + ' ' + ptype

    if any(kw in combined for kw in WATERFALL_KEYWORDS):
        tags.add('waterfall')
        tags.add('strenuous')
    if any(kw in combined for kw in BEACH_KEYWORDS):
        tags.add('beach')
    if any(kw in combined for kw in STRENUOUS_KEYWORDS):
        tags.add('strenuous')
    if any(kw in combined for kw in PARK_KEYWORDS):
        tags.add('park')

    return tags


def _dedupe_route_by_distance_keep_high_wpi(route, threshold_km=NEARBY_DEDUP_KM):
    """
    Keep only one POI in each <=500m pocket for core route,
    preferring higher WPI. Also track nearby alternatives.
    """
    core = []
    nearby = {}

    for poi in route:
        merged = False
        for i, chosen in enumerate(core):
            d = haversine_distance((poi.lat, poi.lon), (chosen.lat, chosen.lon))
            if d <= threshold_km:
                winner, loser = (poi, chosen) if poi.normalized_popularity > chosen.normalized_popularity else (chosen, poi)
                if winner is poi:
                    core[i] = poi
                key = id(winner)
                nearby.setdefault(key, []).append({
                    'name': loser.name,
                    'distance_km': round(d, 3),
                    'wpi': round(loser.normalized_popularity, 3)
                })
                merged = True
                break
        if not merged:
            core.append(poi)

    return core, nearby


def evaluate_fitness(route, start_time=TOUR_START_TIME):
    eval_result = RouteEvaluation()
    if not route:
        eval_result.fitness = 0.0
        return eval_result

    # Apply 500m dedupe for scoring core itinerary quality
    core_route, nearby_alts = _dedupe_route_by_distance_keep_high_wpi(route)
    eval_result.nearby_alternatives = nearby_alts

    current_time_min = time_to_minutes(start_time)
    lunch_start_min = time_to_minutes(LUNCH_START_TIME)
    lunch_end_min = time_to_minutes(LUNCH_END_TIME)

    poi_tags = {id(poi): _classify_poi(poi) for poi in core_route}
    eval_result.waterfall_count = sum(1 for poi in core_route if 'waterfall' in poi_tags[id(poi)])
    eval_result.strenuous_count = sum(1 for poi in core_route if 'strenuous' in poi_tags[id(poi)])

    segment_distances = []

    for i, poi in enumerate(core_route):
        tags = poi_tags[id(poi)]

        if i > 0:
            prev = core_route[i - 1]
            dist_km = get_cached_distance(prev, poi)
            travel_min = calculate_travel_time(prev, poi)
            eval_result.total_distance_km += dist_km
            eval_result.total_travel_time_min += travel_min
            eval_result.distance_penalty += (
                DISTANCE_PENALTY_WEIGHT * dist_km + DISTANCE_QUADRATIC_WEIGHT * (dist_km ** 2)
            )
            segment_distances.append(dist_km)
            current_time_min += travel_min

        if lunch_start_min <= current_time_min < lunch_end_min:
            eval_result.lunch_invasion_count += 1
            eval_result.hard_violation_penalty += LUNCH_INVASION_PENALTY
            current_time_min = lunch_end_min

        arrival_time_min = current_time_min
        poi_open_min = time_to_minutes(poi.opening_time)
        poi_close_min = time_to_minutes(poi.closing_time)

        if current_time_min < poi_open_min:
            current_time_min = poi_open_min
        elif current_time_min >= poi_close_min:
            eval_result.closed_poi_count += 1
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY

        visit_start_min = current_time_min
        visit_end_min = current_time_min + poi.visit_duration_min

        if visit_end_min > poi_close_min and current_time_min < poi_close_min:
            eval_result.hard_violation_penalty += HARD_VIOLATION_PENALTY * 0.5

        eval_result.total_visit_time_min += poi.visit_duration_min
        current_time_min = visit_end_min

        if 'waterfall' in tags:
            if visit_start_min <= time_to_minutes('11:00'):
                eval_result.correct_time_reward += CORRECT_TIME_REWARD
            elif visit_start_min >= AFTERNOON_END_MIN:
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY

        if 'beach' in tags:
            if visit_start_min >= EVENING_START_MIN:
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 1.5
            elif visit_start_min <= time_to_minutes('11:00'):
                eval_result.correct_time_reward += CORRECT_TIME_REWARD * 0.5
            elif MORNING_END_MIN <= visit_start_min < AFTERNOON_END_MIN:
                eval_result.wrong_time_penalty += WRONG_TIME_PENALTY * 0.5

        # New park timing rule (includes amusement + outdoor parks)
        if 'park' in tags:
            if PARK_TIME_START_MIN <= visit_start_min <= PARK_TIME_END_MIN:
                eval_result.correct_time_reward += PARK_CORRECT_TIME_REWARD
            else:
                eval_result.wrong_time_penalty += PARK_WRONG_TIME_PENALTY

        eval_result.timeline.append((
            poi,
            minutes_to_time(int(arrival_time_min)),
            minutes_to_time(int(visit_start_min)),
            minutes_to_time(int(visit_end_min))
        ))

    eval_result.total_time_min = current_time_min - TOUR_START_MINUTES

    if current_time_min > TOUR_END_MINUTES:
        eval_result.overtime_minutes = current_time_min - TOUR_END_MINUTES
        base_over = eval_result.overtime_minutes * OVERTIME_PENALTY_PER_MIN
        if eval_result.overtime_minutes > OVERTIME_HARD_THRESHOLD_MIN:
            base_over *= OVERTIME_STEEP_MULTIPLIER
        eval_result.overtime_penalty = base_over

    undertime_buffer_min = 45
    remaining = TOUR_END_MINUTES - current_time_min
    if remaining > undertime_buffer_min:
        eval_result.undertime_minutes = remaining - undertime_buffer_min
        eval_result.undertime_penalty = eval_result.undertime_minutes * UNDERTIME_PENALTY_PER_MIN

    if eval_result.waterfall_count > 1:
        eval_result.waterfall_penalty = (eval_result.waterfall_count - 1) * EXTRA_WATERFALL_PENALTY

    if eval_result.strenuous_count > 2:
        eval_result.fatigue_penalty = (eval_result.strenuous_count - 2) * PHYSICAL_FATIGUE_PENALTY

    if len(segment_distances) >= 2:
        avg_d = sum(segment_distances) / len(segment_distances)
        peaks = sum(max(0.0, d - avg_d) for d in segment_distances)
        eval_result.wrong_time_penalty += peaks * BACKTRACK_PENALTY_WEIGHT

    # Additional penalty when multiple close-by duplicates still survive in core route
    for i in range(len(core_route)):
        for j in range(i + 1, len(core_route)):
            d = haversine_distance((core_route[i].lat, core_route[i].lon), (core_route[j].lat, core_route[j].lon))
            if d <= NEARBY_DEDUP_KM:
                eval_result.wrong_time_penalty += NEARBY_DUPLICATE_PENALTY

    eval_result.poi_value_sum = sum(p.normalized_popularity for p in core_route)
    time_used = min(eval_result.total_time_min, DAILY_BUDGET_MIN)
    time_ratio = time_used / DAILY_BUDGET_MIN if DAILY_BUDGET_MIN else 0.0
    eval_result.time_utilization_bonus = time_ratio * TIME_UTILIZATION_WEIGHT

    target_pois = 8 if DAILY_BUDGET_MIN >= 480 else 6
    length_gap = abs(len(core_route) - target_pois)
    eval_result.route_length_bonus = max(0.0, 0.6 - 0.08 * length_gap)

    must_see_pois = [p for p in core_route if _is_must_see(p)]
    eval_result.must_see_reward = len(must_see_pois) * MUST_SEE_REWARD

    neighbour_ids = set()
    for anchor in must_see_pois:
        for n in _get_neighbours(anchor, core_route, radius_km=5.0):
            neighbour_ids.add(id(n))
    eval_result.neighbour_reward = len(neighbour_ids) * MUST_SEE_NEIGHBOUR_REWARD

    eval_result.delta = (
        eval_result.distance_penalty
        + eval_result.hard_violation_penalty
        + eval_result.undertime_penalty
        + eval_result.overtime_penalty
        + eval_result.wrong_time_penalty
        + eval_result.waterfall_penalty
        + eval_result.fatigue_penalty
    )

    total_reward = (
        eval_result.poi_value_sum
        + eval_result.time_utilization_bonus
        + eval_result.route_length_bonus
        + eval_result.must_see_reward
        + eval_result.neighbour_reward
        + eval_result.correct_time_reward
    )

    eval_result.fitness = total_reward / (1.0 + eval_result.delta)
    eval_result.travel_penalty = eval_result.distance_penalty
    eval_result.constraint_penalty = eval_result.hard_violation_penalty
    eval_result.user_pref_penalty = 0.0
    eval_result.must_see_penalty = eval_result.waterfall_penalty + eval_result.fatigue_penalty
    eval_result.restaurant_penalty = 0.0

    return eval_result

print('✅ Improved evaluate_fitness() override (v3) loaded: park timing + 500m nearby logic')